# 04a Extraction, batch a: twelve included studies coded from their PDFs

Design log.
- Plan v2 makes notebook 04 the full extraction of the included studies under the frozen schema (schema/record_schema.yaml, v1) and the coding rules (schema/coding_rules.md). The 131 studies left after the pilot are coded in batches of about twelve; each batch notebook writes its records, checks them against the registered PDFs and commits. The last notebook, 04_full_extraction, builds the five CSVs, draws the seeded 20% second-pass sample, scores agreement and lists the decisive values.
- Batch a holds the training and prompting defences most other papers measure against: StruQ (2402.06363), the Instruction Hierarchy (2404.13208), ISE (2410.09102, ICLR 2025), Meta-SecAlign (2507.02735), DefensiveTokens (2507.07974, AISec 2025), Jatmo (2312.17673), BIPIA (2312.14197, KDD 2025), CachePrune (2504.21228), DRIP (2511.00447), FATH (2410.21492), Robustness via Referencing (2504.20472) and Mixture of Encodings (2504.07467).
- Each record says in extraction_scope which sections and table rows were coded: the headline tables and the abstract claims. A quantity outside the scope is never coded as not reported (coding rule 3).
- Every printed value is found on its page by code, in the plain text or in the column-preserving text; every anchor is found on its page. Stage 3 adds three checks the schema module does not make: every record id is unique within its kind, every chart value kept with a figure_only result is also found on its page, and every report is an included study.
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - DRIP trains with preferred responses that process the whole data section, the injected sentence included (its Figure 4 translates the injection into French, and the training prompt adds a do-not-omit instruction). It is coded trains_carry_through yes_by_preference and trains_instruction_pair partly (injected tasks resampled so the same strings also appear as top-level instructions with other data).
  - The Instruction Hierarchy prints its results as bar labels inside charts. They are coded figure_only with the printed value kept, and a residual note says so; the schema has no value_source for a printed chart label.
  - Jatmo compares a fine-tuned davinci-002 with the prompted teacher GPT-3.5-Turbo, so every Jatmo comparison has a different base model and grades B.
  - BIPIA decodes its fine-tuned white-box models with a 512-token cap and the original models with a 2,000-token cap (the no-finetune row repeats Table 3's numbers), so those comparisons grade B.
  - Meta-SecAlign's AgentDojo and InjecAgent rows run with a sandwich prompt in every arm; this is recorded in the evaluation's run field.
  - FATH's Table 3 prints one ASR as a percentage (99.3) and another as a fraction (1.00); each is kept in its printed unit, with a residual note.
- Full-text exclusions found while coding this batch: none.
- Outputs: reports/records/ (twelve files), reports/extraction_04a_check.csv, reports/extraction_04a_summary.csv, reports/extraction_batches.csv (one row per coded report, kept across batches), reports/decision_04a.txt.
- CPU; no network calls; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The twelve records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`struq_arxiv2024` (2402.06363)

In [ ]:
%%writefile reports/records/struq_arxiv2024.yaml
report:
  id: struq_arxiv2024
  slug: '2402.06363'
  title: 'StruQ: Defending Against Prompt Injection with Structured Queries'
  first_author: Sizhe Chen
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2402.06363'
  version_read: arXiv v2
  first_date: '2024-02-09'
  read: F
  role: proposes
  code_release: linked from the abstract
  extraction_scope: 'Sections 5 and 5.1 to 5.4: Llama and Mistral columns of Tables 2 and 3 for the Naive, Completion-Real, Completion-Close (Max), Tree-of-Attack and GCG rows; the abstract claims (other attack rows, HackAPrompt,
    Tables 4 to 9 and Section 5.5 not coded)'
defences:
- id: struq_llama7b
  name: StruQ
  origin: this_report
  base_checkpoint: Llama-7B (base model)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 9
  components:
  - 'secure front-end: reserved special tokens as delimiters, the delimiter strings filtered out of the data'
  - structured instruction tuning on cleaned Alpaca with Naive and Completion-Other injections in the data part
  layout: instruction, data and response sections each opened by reserved delimiter tokens in one prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: code linked; weights not checked
  anchor: We apply StruQ to two popular open-source foundation models
  signal_dependence_detail: Section 5.4 trains separate models with textual or special delimiters (Table 6, attack success 1% in every combination); no run of a trained StruQ model with its delimiters removed or moved
  pages:
    isolation: 11
    signal_dependence: 12
    layout: 7
- id: struq_mistral7b
  name: StruQ
  origin: this_report
  base_checkpoint: Mistral-7B (base model)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 9
  components:
  - 'secure front-end: reserved special tokens as delimiters, the delimiter strings filtered out of the data'
  - structured instruction tuning on cleaned Alpaca with Naive and Completion-Other injections in the data part
  layout: instruction, data and response sections each opened by reserved delimiter tokens in one prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: code linked; weights not checked
  anchor: We apply StruQ to two popular open-source foundation models
  signal_dependence_detail: Section 5.4 trains separate models with textual or special delimiters (Table 6, attack success 1% in every combination); no run of a trained StruQ model with its delimiters removed or moved
  pages:
    isolation: 11
    signal_dependence: 12
    layout: 7
evaluations:
- id: ev_llama_naive_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Naive attack
  injection_position: end
  pages: &id001
    benchmark: 9
    judge: 9
    baseline_kind: 11
- id: ev_llama_naive_def
  defence_id: struq_llama7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Naive attack
  injection_position: end
  pages: *id001
- id: ev_llama_creal_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Completion-Real attack
  injection_position: end
  pages: *id001
- id: ev_llama_creal_def
  defence_id: struq_llama7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Completion-Real attack
  injection_position: end
  pages: *id001
- id: ev_llama_cclose_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: adaptive
  attacker_access: grey_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: highest of nine near-miss delimiter variants (Table 4)
  injection_position: end
  pages: *id001
- id: ev_llama_cclose_def
  defence_id: struq_llama7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: adaptive
  attacker_access: grey_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: highest of nine near-miss delimiter variants (Table 4)
  injection_position: end
  pages: *id001
- id: ev_llama_tap_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Tree-of-Attacks with Pruning, judge LLM replaced by substring matching
  injection_position: end
  pages: *id001
- id: ev_llama_tap_def
  defence_id: struq_llama7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Tree-of-Attacks with Pruning, judge LLM replaced by substring matching
  injection_position: end
  pages: *id001
- id: ev_llama_gcg_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: 500 GCG iterations
  injection_position: end
  pages: *id001
- id: ev_llama_gcg_def
  defence_id: struq_llama7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: 500 GCG iterations
  injection_position: end
  pages: *id001
- id: ev_llama_util_none
  defence_id: none:Llama-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaEval 1.0 (805 AlpacaFarm samples)
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against davinci003
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '805'
  pages: &id002
    benchmark: 9
    judge: 9
    baseline_kind: 11
- id: ev_llama_util_def
  defence_id: struq_llama7b
  benchmark: AlpacaEval 1.0 (805 AlpacaFarm samples)
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against davinci003
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '805'
  pages: *id002
- id: ev_mistral_naive_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Naive attack
  injection_position: end
  pages: &id003
    benchmark: 9
    judge: 9
    baseline_kind: 11
- id: ev_mistral_naive_def
  defence_id: struq_mistral7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Naive attack
  injection_position: end
  pages: *id003
- id: ev_mistral_creal_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Completion-Real attack
  injection_position: end
  pages: *id003
- id: ev_mistral_creal_def
  defence_id: struq_mistral7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: static
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Completion-Real attack
  injection_position: end
  pages: *id003
- id: ev_mistral_cclose_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: adaptive
  attacker_access: grey_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: highest of nine near-miss delimiter variants (Table 4)
  injection_position: end
  pages: *id003
- id: ev_mistral_cclose_def
  defence_id: struq_mistral7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: adaptive
  attacker_access: grey_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: highest of nine near-miss delimiter variants (Table 4)
  injection_position: end
  pages: *id003
- id: ev_mistral_tap_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Tree-of-Attacks with Pruning, judge LLM replaced by substring matching
  injection_position: end
  pages: *id003
- id: ev_mistral_tap_def
  defence_id: struq_mistral7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: black_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: Tree-of-Attacks with Pruning, judge LLM replaced by substring matching
  injection_position: end
  pages: *id003
- id: ev_mistral_gcg_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: 500 GCG iterations
  injection_position: end
  pages: *id003
- id: ev_mistral_gcg_def
  defence_id: struq_mistral7b
  benchmark: AlpacaFarm samples with a data part
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply contains Hacked! or hacked!, each such reply inspected by hand to remove false positives
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '208'
  split: 208 samples, injection at the end of the data
  attacker_budget: 500 GCG iterations
  injection_position: end
  pages: *id003
- id: ev_mistral_util_none
  defence_id: none:Mistral-7B, instruction-tuned on the same Alpaca data (Undef.)
  benchmark: AlpacaEval 1.0 (805 AlpacaFarm samples)
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against davinci003
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  baseline_kind: same_data_twin
  sample_size: '805'
  pages: &id004
    benchmark: 9
    judge: 9
    baseline_kind: 11
- id: ev_mistral_util_def
  defence_id: struq_mistral7b
  benchmark: AlpacaEval 1.0 (805 AlpacaFarm samples)
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against davinci003
  layout_same_as_defence: 'yes'
  measured_by: struq_arxiv2024
  page: 10
  sample_size: '805'
  pages: *id004
results:
- id: r_llama_naive_0_none
  evaluation_id: ev_llama_naive_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Naive
  value: '6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_naive_0_def
  evaluation_id: ev_llama_naive_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Naive
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_creal_0_none
  evaluation_id: ev_llama_creal_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Real
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_creal_0_def
  evaluation_id: ev_llama_creal_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Real
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_cclose_0_none
  evaluation_id: ev_llama_cclose_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Close (Max)
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_cclose_0_def
  evaluation_id: ev_llama_cclose_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Close (Max)
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_tap_0_none
  evaluation_id: ev_llama_tap_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Tree-of-Attack
  value: '97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_tap_0_def
  evaluation_id: ev_llama_tap_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Tree-of-Attack
  value: '9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_gcg_0_none
  evaluation_id: ev_llama_gcg_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Greedy Coordinate Gradient
  value: '97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_gcg_0_def
  evaluation_id: ev_llama_gcg_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Greedy Coordinate Gradient
  value: '58'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_llama_util_0_none
  evaluation_id: ev_llama_util_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (AlpacaEval)
  value: '67.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: AlpacaEval standard error 0.7% (p. 9)
- id: r_llama_util_0_def
  evaluation_id: ev_llama_util_def
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (AlpacaEval)
  value: '67.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: AlpacaEval standard error 0.7% (p. 9)
- id: r_mistral_naive_0_none
  evaluation_id: ev_mistral_naive_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Naive
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_naive_0_def
  evaluation_id: ev_mistral_naive_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Naive
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_creal_0_none
  evaluation_id: ev_mistral_creal_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Real
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_creal_0_def
  evaluation_id: ev_mistral_creal_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Real
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_cclose_0_none
  evaluation_id: ev_mistral_cclose_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Close (Max)
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_cclose_0_def
  evaluation_id: ev_mistral_cclose_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Completion-Close (Max)
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_tap_0_none
  evaluation_id: ev_mistral_tap_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Tree-of-Attack
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_tap_0_def
  evaluation_id: ev_mistral_tap_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Tree-of-Attack
  value: '36'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_gcg_0_none
  evaluation_id: ev_mistral_gcg_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Greedy Coordinate Gradient
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_gcg_0_def
  evaluation_id: ev_mistral_gcg_def
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Greedy Coordinate Gradient
  value: '56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_mistral_util_0_none
  evaluation_id: ev_mistral_util_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (AlpacaEval)
  value: '80.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: AlpacaEval standard error 0.7% (p. 9)
- id: r_mistral_util_0_def
  evaluation_id: ev_mistral_util_def
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (AlpacaEval)
  value: '78.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: AlpacaEval standard error 0.7% (p. 9)
comparisons:
- id: c_llama_naive
  left: ev_llama_naive_def
  right: ev_llama_naive_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_llama_creal
  left: ev_llama_creal_def
  right: ev_llama_creal_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_llama_cclose
  left: ev_llama_cclose_def
  right: ev_llama_cclose_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_llama_tap
  left: ev_llama_tap_def
  right: ev_llama_tap_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_llama_gcg
  left: ev_llama_gcg_def
  right: ev_llama_gcg_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_llama_util
  left: ev_llama_util_def
  right: ev_llama_util_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_naive
  left: ev_mistral_naive_def
  right: ev_mistral_naive_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_creal
  left: ev_mistral_creal_def
  right: ev_mistral_creal_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_cclose
  left: ev_mistral_cclose_def
  right: ev_mistral_cclose_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_tap
  left: ev_mistral_tap_def
  right: ev_mistral_tap_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_gcg
  left: ev_mistral_gcg_def
  right: ev_mistral_gcg_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_mistral_util
  left: ev_mistral_util_def
  right: ev_mistral_util_none
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
claims:
- id: k1
  text: structured queries significantly improve resistance to prompt injection
  location: abstract
  anchor: significantly improves resistance to prompt injection attacks
  jobs:
  - R1
  supported_by:
  - c_llama_naive
  - c_llama_creal
  - c_llama_cclose
  - c_llama_tap
  - c_mistral_naive
  - c_mistral_creal
  - c_mistral_cclose
  - c_mistral_tap
  page: 1
  contradicted_by:
  - r_llama_gcg_0_def
  - r_mistral_gcg_0_def
- id: k2
  text: little or no loss of utility
  location: abstract
  anchor: with little or no impact on utility
  jobs:
  - general_utility
  supported_by:
  - c_llama_util
  - c_mistral_util
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (string match with hand inspection)
    general_utility: Table 3 (AlpacaEval 1.0)

`ih_arxiv2024` (2404.13208)

In [ ]:
%%writefile reports/records/ih_arxiv2024.yaml
report:
  id: ih_arxiv2024
  slug: '2404.13208'
  title: 'The Instruction Hierarchy: Training LLMs to Prioritize Privileged Instructions'
  first_author: Eric Wallace
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2404.13208'
  version_read: arXiv v1
  first_date: '2024-04-19'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'whole report: Sections 3 and 4 and Appendix B; Figure 2 and 3 indirect-injection bars and the Figure 4 User Non-Conflicting Instructions bars coded (direct-injection, extraction and jailbreak bars not
    coded)'
defences:
- id: ih_gpt35
  name: Instruction Hierarchy
  origin: this_report
  base_checkpoint: GPT-3.5 Turbo
  axis1:
  - chat_roles
  axis2:
  - sft
  - preference_optimisation
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - training data that marks instructions as aligned or misaligned by message type (system, user, tool output)
  - targets from context distillation and context ignorance; instructions in browsing and tool outputs treated as misaligned
  - supervised fine-tuning and RLHF on GPT-3.5 Turbo
  layout: system message, user message and tool or browsing outputs as separate message types
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: no (closed model)
  anchor: We fine-tune GPT-3.5 Turbo using supervised finetuning and reinforcement learning
  pages:
    components: 6
    layout_compat: 13
evaluations:
- id: ev_browse_none
  defence_id: none:GPT-3.5 Turbo, fine-tuned on the capability data only (Baseline LM)
  benchmark: 'Prompt Injection (Indirect via Browsing): hand-written injections added to web results'
  attacker_type: human
  attacker_access: black_box
  judge: not stated
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 7
  baseline_kind: same_data_twin
  pages: &id001
    benchmark: 13
    baseline_kind: 7
- id: ev_browse_def
  defence_id: ih_gpt35
  benchmark: 'Prompt Injection (Indirect via Browsing): hand-written injections added to web results'
  attacker_type: human
  attacker_access: black_box
  judge: not stated
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 7
  pages: *id001
- id: ev_tools_none
  defence_id: none:GPT-3.5 Turbo, fine-tuned on the capability data only (Baseline LM)
  benchmark: 'Prompt Injection (Indirect via Tools): injections in tool return values, held-out red-teamer outputs'
  attacker_type: static
  attacker_access: black_box
  judge: not stated
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 7
  baseline_kind: same_data_twin
  attacker_budget: held-out injections written by the RL-trained red-teamer
  pages: *id001
- id: ev_tools_def
  defence_id: ih_gpt35
  benchmark: 'Prompt Injection (Indirect via Tools): injections in tool return values, held-out red-teamer outputs'
  attacker_type: static
  attacker_access: black_box
  judge: not stated
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 7
  attacker_budget: held-out injections written by the RL-trained red-teamer
  pages: *id001
- id: ev_nonconf_none
  defence_id: none:GPT-3.5 Turbo, fine-tuned on the capability data only (Baseline LM)
  benchmark: 'User Non-Conflicting Instructions: held-out synthetic system prompts with reasonable user instructions'
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 judges refusal; non-refusal rate reported
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 9
  baseline_kind: same_data_twin
  pages: &id002
    benchmark: 13
    judge: 13
    baseline_kind: 7
- id: ev_nonconf_def
  defence_id: ih_gpt35
  benchmark: 'User Non-Conflicting Instructions: held-out synthetic system prompts with reasonable user instructions'
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 judges refusal; non-refusal rate reported
  layout_same_as_defence: 'yes'
  measured_by: ih_arxiv2024
  page: 9
  pages: *id002
results:
- id: r_browse_0_none
  evaluation_id: ev_browse_none
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (%)
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 2 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
- id: r_browse_0_def
  evaluation_id: ev_browse_def
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (%)
  value: '85.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 2 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
- id: r_tools_0_none
  evaluation_id: ev_tools_none
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (%)
  value: '77.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
- id: r_tools_0_def
  evaluation_id: ev_tools_def
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (%)
  value: '87.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
- id: r_nonconf_0_none
  evaluation_id: ev_nonconf_none
  quantity: other
  coverage: direct
  metric_as_printed: Compliance Rate (%)
  value: '78.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 9
  location: Figure 4 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
- id: r_nonconf_0_def
  evaluation_id: ev_nonconf_def
  quantity: other
  coverage: direct
  metric_as_printed: Compliance Rate (%)
  value: '77.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 9
  location: Figure 4 bar labels
  spread: error bar of one standard deviation drawn, not printed (p. 7)
comparisons:
- id: c_browse
  left: ev_browse_def
  right: ev_browse_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_tools
  left: ev_tools_def
  right: ev_tools_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_nonconf
  left: ev_nonconf_def
  right: ev_nonconf_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 9
claims:
- id: k1
  text: large gains in robustness, including attack types not seen in training
  location: abstract
  anchor: it drastically increases robustness—even for attack types not seen during training
  jobs:
  - R1
  supported_by:
  - c_browse
  - c_tools
  page: 1
- id: k2
  text: minimal loss of standard capabilities
  location: abstract
  anchor: while imposing minimal degradations on standard capabilities
  jobs:
  - general_utility
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figures 2 and 3 (indirect via browsing, indirect via tools)
    general_utility: Section 4 says capability metrics were comparable; no numbers printed
schema_gaps:
- where: results r_browse_0_*, r_tools_0_*, r_nonconf_0_*
  note: values are printed as bar labels inside the charts; value_source has no value for a printed chart label, so they are coded figure_only and the printed value is kept (checked against the page text by the batch notebook)

`ise_iclr2025` (2410.09102)

In [ ]:
%%writefile reports/records/ise_iclr2025.yaml
report:
  id: ise_iclr2025
  slug: '2410.09102'
  title: 'Instructional Segment Embedding: Improving LLM Safety with Instruction Hierarchy'
  first_author: Tong Wu
  venue: ICLR 2025
  peer_reviewed: 'yes'
  arxiv: '2410.09102'
  version_read: arXiv v2 (marked as published at ICLR 2025)
  first_date: '2024-10-09'
  read: F
  role: proposes
  code_release: PyTorch snippet in Appendix A; repository not checked
  extraction_scope: 'Sections 5 and 6.1: Table 1 (Structured Query benchmark, Llama-2-13B), the AlpacaEval row and the Naive and Completion-R rows for in-domain and out-of-domain attacks; the abstract claims (Ignore and Escape-S
    rows, Average and Worst rows, the Instruction Hierarchy benchmark in Figure 5, Sections 6.3 to 6.4 and the appendix tables not coded)'
defences:
- id: delim_clean_l2_13b
  name: Delimiter (new delimiter tokens, following Chen et al. 2024)
  origin: other_report
  base_checkpoint: Llama-2-13B (pretrained)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - newly added delimiter tokens
  - full supervised fine-tuning on Clean Alpaca
  layout: system and user instructions, data and output separated by newly added delimiter tokens
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not stated
  anchor: we utilize Llama-2-13B (Touvron et al., 2023) and Llama-3-8B
  pages:
    components: 6
- id: ise_clean_l2_13b
  name: ISE
  origin: this_report
  base_checkpoint: Llama-2-13B (pretrained)
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - learned segment embedding per token type (system, user, data, output)
  - full supervised fine-tuning on Clean Alpaca
  layout: text delimiters, with a learned segment embedding added to every token (instruction, data or output)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not stated
  anchor: we utilize Llama-2-13B (Touvron et al., 2023) and Llama-3-8B
  pages:
    isolation: 7
    components: 5
- id: advtext_l2_13b
  name: Adversarial Alpaca training with text delimiters (Baseline)
  origin: other_report
  base_checkpoint: Llama-2-13B (pretrained)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - text delimiters
  - full supervised fine-tuning on Adversarial Alpaca (instructions from other samples injected into the data, plain or after a fake response)
  layout: system and user instructions, data and output separated by text delimiters
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not stated
  anchor: we utilize Llama-2-13B (Touvron et al., 2023) and Llama-3-8B
  pages:
    components: 6
- id: delim_adv_l2_13b
  name: Delimiter (new delimiter tokens, following Chen et al. 2024)
  origin: other_report
  base_checkpoint: Llama-2-13B (pretrained)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - newly added delimiter tokens
  - full supervised fine-tuning on Adversarial Alpaca (instructions from other samples injected into the data, plain or after a fake response)
  layout: system and user instructions, data and output separated by newly added delimiter tokens
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not stated
  anchor: we utilize Llama-2-13B (Touvron et al., 2023) and Llama-3-8B
  pages:
    components: 6
- id: ise_adv_l2_13b
  name: ISE
  origin: this_report
  base_checkpoint: Llama-2-13B (pretrained)
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - learned segment embedding per token type (system, user, data, output)
  - full supervised fine-tuning on Adversarial Alpaca (instructions from other samples injected into the data, plain or after a fake response)
  layout: text delimiters, with a learned segment embedding added to every token (instruction, data or output)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not stated
  anchor: we utilize Llama-2-13B (Touvron et al., 2023) and Llama-3-8B
  pages:
    isolation: 7
    components: 5
evaluations:
- id: ev_base_clean_util
  defence_id: none:Llama-2-13B, instruction-tuned on Clean Alpaca with text delimiters (Baseline)
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
  baseline_kind: same_data_twin
- id: ev_delim_clean_util
  defence_id: delim_clean_l2_13b
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
- id: ev_ise_clean_util
  defence_id: ise_clean_l2_13b
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
- id: ev_base_adv_util
  defence_id: advtext_l2_13b
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
- id: ev_delim_adv_util
  defence_id: delim_adv_l2_13b
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
- id: ev_ise_adv_util
  defence_id: ise_adv_l2_13b
  benchmark: AlpacaEval 1.0
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4 pairwise preference against the reference model
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  decoding: top-p sampling with the model's default settings
  pages:
    benchmark: 6
    decoding: 5
- id: ev_base_clean_id_naive
  defence_id: none:Llama-2-13B, instruction-tuned on Clean Alpaca with text delimiters (Baseline)
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: &id001
    benchmark: 6
    judge: 6
    decoding: 5
    sample_size: 19
  baseline_kind: same_data_twin
- id: ev_delim_clean_id_naive
  defence_id: delim_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_clean_id_naive
  defence_id: ise_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_adv_id_naive
  defence_id: advtext_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_delim_adv_id_naive
  defence_id: delim_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_adv_id_naive
  defence_id: ise_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Naive
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_clean_id_creal
  defence_id: none:Llama-2-13B, instruction-tuned on Clean Alpaca with text delimiters (Baseline)
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
  baseline_kind: same_data_twin
- id: ev_delim_clean_id_creal
  defence_id: delim_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_clean_id_creal
  defence_id: ise_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_adv_id_creal
  defence_id: advtext_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_delim_adv_id_creal
  defence_id: delim_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_adv_id_creal
  defence_id: ise_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: In-Domain Completion-R
  injection_position: end
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_clean_ood_naive
  defence_id: none:Llama-2-13B, instruction-tuned on Clean Alpaca with text delimiters (Baseline)
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
  baseline_kind: same_data_twin
- id: ev_delim_clean_ood_naive
  defence_id: delim_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_clean_ood_naive
  defence_id: ise_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_adv_ood_naive
  defence_id: advtext_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_delim_adv_ood_naive
  defence_id: delim_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_adv_ood_naive
  defence_id: ise_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Naive
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_clean_ood_creal
  defence_id: none:Llama-2-13B, instruction-tuned on Clean Alpaca with text delimiters (Baseline)
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
  baseline_kind: same_data_twin
- id: ev_delim_clean_ood_creal
  defence_id: delim_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_clean_ood_creal
  defence_id: ise_clean_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_base_adv_ood_creal
  defence_id: advtext_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_delim_adv_ood_creal
  defence_id: delim_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
- id: ev_ise_adv_ood_creal
  defence_id: ise_adv_l2_13b
  benchmark: AlpacaEval test samples with a data part (Structured Query benchmark)
  attacker_type: static
  attacker_access: black_box
  judge: reply does not contain the word hacked
  layout_same_as_defence: 'yes'
  measured_by: ise_iclr2025
  page: 7
  split: 208 samples
  sample_size: '208'
  attacker_budget: Out-of-Domain Completion-R
  injection_position: mixed
  decoding: top-p sampling with the model's default settings
  pages: *id001
results:
- id: r_base_clean_util
  evaluation_id: ev_base_clean_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '72.76'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_clean_util
  evaluation_id: ev_delim_clean_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '72.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_clean_util
  evaluation_id: ev_ise_clean_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '72.13'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_adv_util
  evaluation_id: ev_base_adv_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '73.41'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_adv_util
  evaluation_id: ev_delim_adv_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '72.26'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_adv_util
  evaluation_id: ev_ise_adv_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability (AlpacaEval)
  value: '73.76'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_clean_id_naive
  evaluation_id: ev_base_clean_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '65.87'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_clean_id_naive
  evaluation_id: ev_delim_clean_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '68.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_clean_id_naive
  evaluation_id: ev_ise_clean_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '75.96'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_adv_id_naive
  evaluation_id: ev_base_adv_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_adv_id_naive
  evaluation_id: ev_delim_adv_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '99.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_adv_id_naive
  evaluation_id: ev_ise_adv_id_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Naive)
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_clean_id_creal
  evaluation_id: ev_base_clean_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '4.81'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_clean_id_creal
  evaluation_id: ev_delim_clean_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '7.21'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_clean_id_creal
  evaluation_id: ev_ise_clean_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '40.38'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_adv_id_creal
  evaluation_id: ev_base_adv_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '70.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_adv_id_creal
  evaluation_id: ev_delim_adv_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_adv_id_creal
  evaluation_id: ev_ise_adv_id_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, In-Domain Completion-R)
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_clean_ood_naive
  evaluation_id: ev_base_clean_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '62.02'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_clean_ood_naive
  evaluation_id: ev_delim_clean_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '66.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_clean_ood_naive
  evaluation_id: ev_ise_clean_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '69.71'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_adv_ood_naive
  evaluation_id: ev_base_adv_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '64.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_adv_ood_naive
  evaluation_id: ev_delim_adv_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '67.79'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_adv_ood_naive
  evaluation_id: ev_ise_adv_ood_naive
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Naive)
  value: '76.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_clean_ood_creal
  evaluation_id: ev_base_clean_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '1.92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_clean_ood_creal
  evaluation_id: ev_delim_clean_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '12.99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_clean_ood_creal
  evaluation_id: ev_ise_clean_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '34.14'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_base_adv_ood_creal
  evaluation_id: ev_base_adv_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '85.58'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_delim_adv_ood_creal
  evaluation_id: ev_delim_adv_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '91.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_ise_adv_ood_creal
  evaluation_id: ev_ise_adv_ood_creal
  quantity: R1
  coverage: direct
  metric_as_printed: Robustness (Robustness, Out-of-Domain Completion-R)
  value: '99.52'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
comparisons:
- id: c_util_ise_clean_base_clean
  left: ev_ise_clean_util
  right: ev_base_clean_util
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same-data twin without it, both trained on Clean Alpaca
  page: 7
- id: c_util_ise_adv_base_adv
  left: ev_ise_adv_util
  right: ev_base_adv_util
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same training without it, both trained on Adversarial Alpaca
  page: 7
- id: c_util_ise_adv_delim_adv
  left: ev_ise_adv_util
  right: ev_delim_adv_util
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: ISE against new delimiter tokens (the StruQ-style design), both trained on Adversarial Alpaca
  page: 7
- id: c_id_naive_ise_clean_base_clean
  left: ev_ise_clean_id_naive
  right: ev_base_clean_id_naive
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same-data twin without it, both trained on Clean Alpaca
  page: 7
- id: c_id_naive_ise_adv_base_adv
  left: ev_ise_adv_id_naive
  right: ev_base_adv_id_naive
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same training without it, both trained on Adversarial Alpaca
  page: 7
- id: c_id_naive_ise_adv_delim_adv
  left: ev_ise_adv_id_naive
  right: ev_delim_adv_id_naive
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: ISE against new delimiter tokens (the StruQ-style design), both trained on Adversarial Alpaca
  page: 7
- id: c_id_creal_ise_clean_base_clean
  left: ev_ise_clean_id_creal
  right: ev_base_clean_id_creal
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same-data twin without it, both trained on Clean Alpaca
  page: 7
- id: c_id_creal_ise_adv_base_adv
  left: ev_ise_adv_id_creal
  right: ev_base_adv_id_creal
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same training without it, both trained on Adversarial Alpaca
  page: 7
- id: c_id_creal_ise_adv_delim_adv
  left: ev_ise_adv_id_creal
  right: ev_delim_adv_id_creal
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: ISE against new delimiter tokens (the StruQ-style design), both trained on Adversarial Alpaca
  page: 7
- id: c_ood_naive_ise_clean_base_clean
  left: ev_ise_clean_ood_naive
  right: ev_base_clean_ood_naive
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same-data twin without it, both trained on Clean Alpaca
  page: 7
- id: c_ood_naive_ise_adv_base_adv
  left: ev_ise_adv_ood_naive
  right: ev_base_adv_ood_naive
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same training without it, both trained on Adversarial Alpaca
  page: 7
- id: c_ood_naive_ise_adv_delim_adv
  left: ev_ise_adv_ood_naive
  right: ev_delim_adv_ood_naive
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: ISE against new delimiter tokens (the StruQ-style design), both trained on Adversarial Alpaca
  page: 7
- id: c_ood_creal_ise_clean_base_clean
  left: ev_ise_clean_ood_creal
  right: ev_base_clean_ood_creal
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same-data twin without it, both trained on Clean Alpaca
  page: 7
- id: c_ood_creal_ise_adv_base_adv
  left: ev_ise_adv_ood_creal
  right: ev_base_adv_ood_creal
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the learned segment embedding (ISE) against the same training without it, both trained on Adversarial Alpaca
  page: 7
- id: c_ood_creal_ise_adv_delim_adv
  left: ev_ise_adv_ood_creal
  right: ev_delim_adv_ood_creal
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: ISE against new delimiter tokens (the StruQ-style design), both trained on Adversarial Alpaca
  page: 7
claims:
- id: k1
  text: average robust accuracy rises by up to 15.75% (Structured Query) and 18.68% (Instruction Hierarchy)
  location: abstract
  anchor: an average robust accuracy increase of up to 15.75% and 18.68%
  jobs:
  - R1
  supported_by:
  - c_id_naive_ise_clean_base_clean
  - c_id_creal_ise_clean_base_clean
  page: 1
- id: k2
  text: instruction following on AlpacaEval improves by up to 4.1%
  location: abstract
  anchor: an improvement in the instruction-following capability of up to 4.1% on AlpacaEval
  jobs:
  - general_utility
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (string match on hacked)
    general_utility: Table 1 (AlpacaEval 1.0)

`metasecalign_arxiv2025` (2507.02735)

In [ ]:
%%writefile reports/records/metasecalign_arxiv2025.yaml
report:
  id: metasecalign_arxiv2025
  slug: '2507.02735'
  title: 'Meta-SecAlign: Training LLMs against Prompt Injection for Robust Agents'
  first_author: Sizhe Chen
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2507.02735'
  version_read: arXiv v4
  first_date: '2025-07-03'
  read: F
  role: proposes
  code_release: training and evaluation code stated as public (p. 2); not checked
  extraction_scope: 'Sections 3 and 4.1 to 4.3 and Appendices A and B: the Llama-3.3-70B-Instruct Undef., SecAlign and Ours columns of Tables 3 and 4, the GCG row of Table 5 (8B and 70B), and the randomized-position columns
    of Table 1 for AgentDojo utility and AlpacaFarm ASR; the abstract claims (Table 2, commercial-model columns, Table 6, Appendices C to F not coded)'
defences:
- id: msa_70b
  name: Meta-SecAlign
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  name_as_printed: Meta-SecAlign-70B
  components:
  - added input role for the data
  - SecAlign simulated injections at a randomized position
  - self-generated preferred and rejected responses from the initial model
  - DPO with LoRA on Cleaned-Alpaca (19157 pairs)
  layout: instruction in the user role, data in an added input role, Llama chat template
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: defended models stated as released under commercial licences (p. 2)
  anchor: we fine-tune L LAMA -3.1-8B-I NSTRUCT and L LAMA -3.3-70B
  signal_dependence_detail: models must be used with the exact prompt format (p. 15); no numbers
  pages:
    components: 15
    isolation: 5
    signal_dependence: 15
    layout: 15
    released_weights: 2
- id: secalign_70b
  name: SecAlign (re-trained by the authors with the Meta-SecAlign chat template)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: instruction in the user role, data in an added input role, Llama chat template
  anchor: Our SecAlign LLMs use the same chat template as Meta-SecAlign models
- id: msa_8b
  name: Meta-SecAlign
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  name_as_printed: Meta-SecAlign-8B
  components:
  - added input role for the data
  - SecAlign simulated injections at a randomized position
  - self-generated preferred and rejected responses from the initial model
  - DPO with LoRA on Cleaned-Alpaca (19157 pairs)
  layout: instruction in the user role, data in an added input role, Llama chat template
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: defended models stated as released under commercial licences (p. 2)
  anchor: we fine-tune L LAMA -3.1-8B-I NSTRUCT and L LAMA -3.3-70B
  signal_dependence_detail: models must be used with the exact prompt format (p. 15); no numbers
  pages:
    components: 15
    isolation: 5
    signal_dependence: 15
    layout: 15
    released_weights: 2
- id: secalign_8b
  name: SecAlign (re-trained by the authors with the Meta-SecAlign chat template)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: instruction in the user role, data in an added input role, Llama chat template
  anchor: Our SecAlign LLMs use the same chat template as Meta-SecAlign models
- id: msa_fixedpos_70b
  name: Meta-SecAlign without randomized injection position
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  layout: instruction in the user role, data in an added input role, Llama chat template
  anchor: Randomized injection position improves utility while maintaining similar security
evaluations:
- id: ev_70b_none_alpacaeval
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  sample_size: '805'
  decoding: temperature 0 (Table 7)
  pages: &id001
    benchmark: 16
    judge: 16
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_70b_sec_alpacaeval
  defence_id: secalign_70b
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  sample_size: '805'
  decoding: temperature 0 (Table 7)
  pages: *id001
- id: ev_70b_msa_alpacaeval
  defence_id: msa_70b
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  sample_size: '805'
  decoding: temperature 0 (Table 7)
  pages: *id001
- id: ev_70b_none_seputil
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: SEP samples, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  decoding: temperature 0 (Table 7)
  pages: &id002
    benchmark: 16
    judge: 16
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_70b_sec_seputil
  defence_id: secalign_70b
  benchmark: SEP samples, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  decoding: temperature 0 (Table 7)
  pages: *id002
- id: ev_70b_msa_seputil
  defence_id: msa_70b
  benchmark: SEP samples, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  decoding: temperature 0 (Table 7)
  pages: *id002
- id: ev_70b_none_alpacafarm
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked!
  attacker_type: static
  attacker_access: black_box
  judge: reply begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of 6 attacks (straightforward and ignore before or after data, completion, completion-ignore)
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: &id003
    benchmark: 16
    judge: 16
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_70b_sec_alpacafarm
  defence_id: secalign_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked!
  attacker_type: static
  attacker_access: black_box
  judge: reply begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of 6 attacks (straightforward and ignore before or after data, completion, completion-ignore)
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id003
- id: ev_70b_msa_alpacafarm
  defence_id: msa_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked!
  attacker_type: static
  attacker_access: black_box
  judge: reply begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of 6 attacks (straightforward and ignore before or after data, completion, completion-ignore)
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id003
- id: ev_70b_none_sepasr
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: SEP (9.1K samples, one injected instruction each)
  attacker_type: static
  attacker_access: black_box
  judge: witness word appears in the reply
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of the same 6 attacks
  decoding: temperature 0 (Table 7)
  pages: &id004
    benchmark: 16
    judge: 16
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_70b_sec_sepasr
  defence_id: secalign_70b
  benchmark: SEP (9.1K samples, one injected instruction each)
  attacker_type: static
  attacker_access: black_box
  judge: witness word appears in the reply
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of the same 6 attacks
  decoding: temperature 0 (Table 7)
  pages: *id004
- id: ev_70b_msa_sepasr
  defence_id: msa_70b
  benchmark: SEP (9.1K samples, one injected instruction each)
  attacker_type: static
  attacker_access: black_box
  judge: witness word appears in the reply
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: any of the same 6 attacks
  decoding: temperature 0 (Table 7)
  pages: *id004
- id: ev_70b_none_tasktracker
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: TaskTracker (31K samples)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the reply responds to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: injection positions and ignore sentences as specified by the dataset
  decoding: temperature 0 (Table 7)
  pages: &id005
    benchmark: 16
    judge: 17
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_70b_sec_tasktracker
  defence_id: secalign_70b
  benchmark: TaskTracker (31K samples)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the reply responds to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: injection positions and ignore sentences as specified by the dataset
  decoding: temperature 0 (Table 7)
  pages: *id005
- id: ev_70b_msa_tasktracker
  defence_id: msa_70b
  benchmark: TaskTracker (31K samples)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the reply responds to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 7
  attacker_budget: injection positions and ignore sentences as specified by the dataset
  decoding: temperature 0 (Table 7)
  pages: *id005
- id: ev_70b_none_dojo_clean
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker (user goal achieved)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '97'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: &id006
    benchmark: 17
    judge: 17
    decoding: 16
    run: 17
  baseline_kind: off_the_shelf
- id: ev_70b_sec_dojo_clean
  defence_id: secalign_70b
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker (user goal achieved)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '97'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id006
- id: ev_70b_msa_dojo_clean
  defence_id: msa_70b
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker (user goal achieved)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '97'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id006
- id: ev_70b_none_dojo_attack
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, 949 user-task and injection-task pairs
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checkers: user goal achieved (utility under attack); malicious API called (ASR)'
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: important_instructions attack
  sample_size: '949'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: &id007
    benchmark: 17
    judge: 17
    decoding: 16
    run: 17
  baseline_kind: off_the_shelf
- id: ev_70b_sec_dojo_attack
  defence_id: secalign_70b
  benchmark: AgentDojo, 949 user-task and injection-task pairs
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checkers: user goal achieved (utility under attack); malicious API called (ASR)'
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: important_instructions attack
  sample_size: '949'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id007
- id: ev_70b_msa_dojo_attack
  defence_id: msa_70b
  benchmark: AgentDojo, 949 user-task and injection-task pairs
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checkers: user goal achieved (utility under attack); malicious API called (ASR)'
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: important_instructions attack
  sample_size: '949'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id007
- id: ev_70b_none_wasp_clean
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: WASP benign utility set (WebArena, axtree pages)
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '37'
  decoding: not reported
  pages: &id008
    benchmark: 17
    judge: 17
  baseline_kind: off_the_shelf
- id: ev_70b_sec_wasp_clean
  defence_id: secalign_70b
  benchmark: WASP benign utility set (WebArena, axtree pages)
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '37'
  decoding: not reported
  pages: *id008
- id: ev_70b_msa_wasp_clean
  defence_id: msa_70b
  benchmark: WASP benign utility set (WebArena, axtree pages)
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '37'
  decoding: not reported
  pages: *id008
- id: ev_70b_none_wasp_attack
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: WASP injected samples (2 benign tasks x 42 injected tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent diverted at any point (intermediate) or injected task completed (end to end)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '84'
  decoding: not reported
  pages: &id009
    benchmark: 17
    judge: 17
  baseline_kind: off_the_shelf
- id: ev_70b_sec_wasp_attack
  defence_id: secalign_70b
  benchmark: WASP injected samples (2 benign tasks x 42 injected tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent diverted at any point (intermediate) or injected task completed (end to end)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '84'
  decoding: not reported
  pages: *id009
- id: ev_70b_msa_wasp_attack
  defence_id: msa_70b
  benchmark: WASP injected samples (2 benign tasks x 42 injected tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent diverted at any point (intermediate) or injected task completed (end to end)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  sample_size: '84'
  decoding: not reported
  pages: *id009
- id: ev_70b_none_injecagent
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: InjecAgent (1K API-calling samples)
  attacker_type: static
  attacker_access: black_box
  judge: the malicious API is called (ASR-total)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: higher of base and enhanced settings
  sample_size: 1K
  decoding: temperature 0 (Table 7)
  run: with the sandwich defence in every arm
  pages: &id010
    benchmark: 17
    judge: 17
    decoding: 16
    run: 17
  baseline_kind: off_the_shelf
- id: ev_70b_sec_injecagent
  defence_id: secalign_70b
  benchmark: InjecAgent (1K API-calling samples)
  attacker_type: static
  attacker_access: black_box
  judge: the malicious API is called (ASR-total)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: higher of base and enhanced settings
  sample_size: 1K
  decoding: temperature 0 (Table 7)
  run: with the sandwich defence in every arm
  pages: *id010
- id: ev_70b_msa_injecagent
  defence_id: msa_70b
  benchmark: InjecAgent (1K API-calling samples)
  attacker_type: static
  attacker_access: black_box
  judge: the malicious API is called (ASR-total)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: higher of base and enhanced settings
  sample_size: 1K
  decoding: temperature 0 (Table 7)
  run: with the sandwich defence in every arm
  pages: *id010
- id: ev_8b_none_gcg
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: &id011
    benchmark: 16
    judge: 16
    attacker_budget: 16
    decoding: 16
  baseline_kind: off_the_shelf
- id: ev_8b_sec_gcg
  defence_id: secalign_8b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id011
- id: ev_8b_msa_gcg
  defence_id: msa_8b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id011
- id: ev_70b_none_gcg
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id011
  baseline_kind: off_the_shelf
- id: ev_70b_sec_gcg
  defence_id: secalign_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id011
- id: ev_70b_msa_gcg
  defence_id: msa_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked, ignoring case and surrounding whitespace
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 8
  attacker_budget: 500 GCG steps per example, 512 candidates per step, one run, no restarts
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id011
- id: ev_t1_fixed_dojo_clean
  defence_id: msa_fixedpos_70b
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker (user goal achieved)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 5
  sample_size: '97'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id006
- id: ev_t1_rand_dojo_clean
  defence_id: msa_70b
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker (user goal achieved)
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 5
  sample_size: '97'
  decoding: temperature 0, top-p 0.9, 16K context
  run: with AgentDojo's repeat_user_prompt (sandwich) defence and the Llama system prompt in every arm
  pages: *id006
- id: ev_t1_fixed_alpacafarm
  defence_id: msa_fixedpos_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked!
  attacker_type: static
  attacker_access: black_box
  judge: reply begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 5
  attacker_budget: any of 6 attacks (straightforward and ignore before or after data, completion, completion-ignore)
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id003
- id: ev_t1_rand_alpacafarm
  defence_id: msa_70b
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked!
  attacker_type: static
  attacker_access: black_box
  judge: reply begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: metasecalign_arxiv2025
  page: 5
  attacker_budget: any of 6 attacks (straightforward and ignore before or after data, completion, completion-ignore)
  sample_size: '208'
  decoding: temperature 0 (Table 7)
  pages: *id003
results:
- id: r_70b_none_alpacaeval2_utility
  evaluation_id: ev_70b_none_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 Utility
  value: '44.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_sec_alpacaeval2_utility
  evaluation_id: ev_70b_sec_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 Utility
  value: '38.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_msa_alpacaeval2_utility
  evaluation_id: ev_70b_msa_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 Utility
  value: '44.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_none_sep_utility
  evaluation_id: ev_70b_none_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility
  value: '62.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_sec_sep_utility
  evaluation_id: ev_70b_sec_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility
  value: '51.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_msa_sep_utility
  evaluation_id: ev_70b_msa_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility
  value: '60.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_none_alpacafarm_asr
  evaluation_id: ev_70b_none_alpacafarm
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '95.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_sec_alpacafarm_asr
  evaluation_id: ev_70b_sec_alpacafarm
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_msa_alpacafarm_asr
  evaluation_id: ev_70b_msa_alpacafarm
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_none_sep_asr
  evaluation_id: ev_70b_none_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '99.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_sec_sep_asr
  evaluation_id: ev_70b_sec_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '12.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_msa_sep_asr
  evaluation_id: ev_70b_msa_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_none_tasktracker_asr
  evaluation_id: ev_70b_none_tasktracker
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '19.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_sec_tasktracker_asr
  evaluation_id: ev_70b_sec_tasktracker
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_msa_tasktracker_asr
  evaluation_id: ev_70b_msa_tasktracker
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_70b_none_agentdojo_utility
  evaluation_id: ev_70b_none_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo Utility
  value: '59.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_agentdojo_utility
  evaluation_id: ev_70b_sec_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo Utility
  value: '6.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_agentdojo_utility
  evaluation_id: ev_70b_msa_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo Utility
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_agentdojo_utility_w_attack
  evaluation_id: ev_70b_none_dojo_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Utility w. Attack
  value: '43.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_agentdojo_utility_w_attack
  evaluation_id: ev_70b_sec_dojo_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Utility w. Attack
  value: '7.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_agentdojo_utility_w_attack
  evaluation_id: ev_70b_msa_dojo_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Utility w. Attack
  value: '79.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_agentdojo_asr
  evaluation_id: ev_70b_none_dojo_attack
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '14.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_agentdojo_asr
  evaluation_id: ev_70b_sec_dojo_attack
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_agentdojo_asr
  evaluation_id: ev_70b_msa_dojo_attack
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_wasp_utility
  evaluation_id: ev_70b_none_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP Utility
  value: '62.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_wasp_utility
  evaluation_id: ev_70b_sec_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP Utility
  value: '45.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_wasp_utility
  evaluation_id: ev_70b_msa_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP Utility
  value: '59.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_wasp_intermediate_asr
  evaluation_id: ev_70b_none_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP Intermediate ASR
  value: '20.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_wasp_intermediate_asr
  evaluation_id: ev_70b_sec_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP Intermediate ASR
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_wasp_intermediate_asr
  evaluation_id: ev_70b_msa_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP Intermediate ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_wasp_end2end_asr
  evaluation_id: ev_70b_none_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP End2End ASR
  value: '2.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_wasp_end2end_asr
  evaluation_id: ev_70b_sec_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP End2End ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_wasp_end2end_asr
  evaluation_id: ev_70b_msa_wasp_attack
  quantity: R1
  coverage: direct
  metric_as_printed: WASP End2End ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_none_injecagent_asr
  evaluation_id: ev_70b_none_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '53.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_sec_injecagent_asr
  evaluation_id: ev_70b_sec_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_70b_msa_injecagent_asr
  evaluation_id: ev_70b_msa_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_8b_none_gcg
  evaluation_id: ev_8b_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '87.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_8b_sec_gcg
  evaluation_id: ev_8b_sec_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '28.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_8b_msa_gcg
  evaluation_id: ev_8b_msa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '20.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_70b_none_gcg
  evaluation_id: ev_70b_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '98.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_70b_sec_gcg
  evaluation_id: ev_70b_sec_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '53.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_70b_msa_gcg
  evaluation_id: ev_70b_msa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG Adaptive ASR
  value: '47.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t1_fixed_dojo_clean
  evaluation_id: ev_t1_fixed_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo Utility
  value: '15.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_rand_dojo_clean
  evaluation_id: ev_t1_rand_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo Utility
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_fixed_alpacafarm
  evaluation_id: ev_t1_fixed_alpacafarm
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_rand_alpacafarm
  evaluation_id: ev_t1_rand_alpacafarm
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
comparisons:
- id: c_70b_alpacaeval_msa_none
  left: ev_70b_msa_alpacaeval
  right: ev_70b_none_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_70b_alpacaeval_msa_sec
  left: ev_70b_msa_alpacaeval
  right: ev_70b_sec_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 7
- id: c_70b_seputil_msa_none
  left: ev_70b_msa_seputil
  right: ev_70b_none_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_70b_seputil_msa_sec
  left: ev_70b_msa_seputil
  right: ev_70b_sec_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 7
- id: c_70b_alpacafarm_msa_none
  left: ev_70b_msa_alpacafarm
  right: ev_70b_none_alpacafarm
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_70b_alpacafarm_msa_sec
  left: ev_70b_msa_alpacafarm
  right: ev_70b_sec_alpacafarm
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 7
- id: c_70b_sepasr_msa_none
  left: ev_70b_msa_sepasr
  right: ev_70b_none_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_70b_sepasr_msa_sec
  left: ev_70b_msa_sepasr
  right: ev_70b_sec_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 7
- id: c_70b_tasktracker_msa_none
  left: ev_70b_msa_tasktracker
  right: ev_70b_none_tasktracker
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_70b_tasktracker_msa_sec
  left: ev_70b_msa_tasktracker
  right: ev_70b_sec_tasktracker
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 7
- id: c_70b_dojo_clean_msa_none
  left: ev_70b_msa_dojo_clean
  right: ev_70b_none_dojo_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_dojo_clean_msa_sec
  left: ev_70b_msa_dojo_clean
  right: ev_70b_sec_dojo_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_70b_dojo_attack_msa_none
  left: ev_70b_msa_dojo_attack
  right: ev_70b_none_dojo_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_dojo_attack_msa_sec
  left: ev_70b_msa_dojo_attack
  right: ev_70b_sec_dojo_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_70b_wasp_clean_msa_none
  left: ev_70b_msa_wasp_clean
  right: ev_70b_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_wasp_clean_msa_sec
  left: ev_70b_msa_wasp_clean
  right: ev_70b_sec_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_70b_wasp_attack_msa_none
  left: ev_70b_msa_wasp_attack
  right: ev_70b_none_wasp_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_wasp_attack_msa_sec
  left: ev_70b_msa_wasp_attack
  right: ev_70b_sec_wasp_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_70b_injecagent_msa_none
  left: ev_70b_msa_injecagent
  right: ev_70b_none_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_injecagent_msa_sec
  left: ev_70b_msa_injecagent
  right: ev_70b_sec_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_8b_gcg_msa_none
  left: ev_8b_msa_gcg
  right: ev_8b_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_8b_gcg_msa_sec
  left: ev_8b_msa_gcg
  right: ev_8b_sec_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_70b_gcg_msa_none
  left: ev_70b_msa_gcg
  right: ev_70b_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_70b_gcg_msa_sec
  left: ev_70b_msa_gcg
  right: ev_70b_sec_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Meta-SecAlign against SecAlign re-trained with the same chat template'
  page: 8
- id: c_t1_dojo_clean
  left: ev_t1_rand_dojo_clean
  right: ev_t1_fixed_dojo_clean
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: randomized injection position in training (on against off), self-generated labels in both
  page: 5
- id: c_t1_alpacafarm
  left: ev_t1_rand_alpacafarm
  right: ev_t1_fixed_alpacafarm
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: randomized injection position in training (on against off), self-generated labels in both
  page: 5
claims:
- id: k1
  text: Meta-SecAlign keeps almost all of the undefended model's utility
  location: abstract
  anchor: Meta-SecAlign maintains almost all the undefended LLM’s utility
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_70b_alpacaeval_msa_none
  - c_70b_seputil_msa_none
  - c_70b_dojo_clean_msa_none
  - c_70b_dojo_attack_msa_none
  - c_70b_wasp_clean_msa_none
  page: 1
- id: k2
  text: better overall security than SecAlign against static and GCG adaptive attacks
  location: abstract
  anchor: achieving better overall security than SecAlign against various static and GCG adaptive attacks
  jobs:
  - R1
  supported_by:
  - c_70b_sepasr_msa_sec
  - c_70b_wasp_attack_msa_sec
  - c_8b_gcg_msa_sec
  - c_70b_gcg_msa_sec
  page: 1
  contradicted_by:
  - r_70b_msa_injecagent_asr
  - r_70b_msa_agentdojo_asr
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 3, 4 and 5
    R2clean: Table 4 (AgentDojo and WASP task checkers); SEP utility is a proxy
    R2attack: Table 4 (AgentDojo utility under attack)
    general_utility: Table 3 (AlpacaEval2)

`defensivetokens_aisec2025` (2507.07974)

In [ ]:
%%writefile reports/records/defensivetokens_aisec2025.yaml
report:
  id: defensivetokens_aisec2025
  slug: '2507.07974'
  title: Defending Against Prompt Injection With a Few DefensiveTokens
  first_author: Sizhe Chen
  venue: AISec 2025 (ACM Workshop on Artificial Intelligence and Security)
  peer_reviewed: 'yes'
  doi: 10.1145/3733799.3762982
  arxiv: '2507.07974'
  version_read: arXiv v2 (carries the AISec '25 header and DOI)
  first_date: '2025-07-10'
  read: F
  role: proposes
  code_release: linked from the abstract
  extraction_scope: 'Sections 3 and 4.1 to 4.4: the None and DefensiveToken rows of Table 3 for all four models and all eight columns, and the SecAlign-LoRA row for Llama3.1-8B-Instruct; the abstract claims (TextGrad, Reminder,
    Sandwich and StruQ rows, Figures 2 and 3, Tables 4 to 8 not coded)'
defences:
- id: dt_l3
  name: DefensiveToken
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 5 new token embeddings prepended to the input, optimised with the model frozen
  - StruQ loss and StruQ defensive dataset from Cleaned Alpaca, labels generated by the undefended model
  layout: instruction in the system role, data in the user role, response in the assistant role, DefensiveTokens before the input
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; token embeddings not checked
  anchor: We optimize 5 defensive tokens, placed before the LLM input
  pages:
    components: 3
    isolation: 7
- id: dt_l31
  name: DefensiveToken
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 5 new token embeddings prepended to the input, optimised with the model frozen
  - StruQ loss and StruQ defensive dataset from Cleaned Alpaca, labels generated by the undefended model
  layout: instruction in the system role, data in the user role, response in the assistant role, DefensiveTokens before the input
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; token embeddings not checked
  anchor: We optimize 5 defensive tokens, placed before the LLM input
  pages:
    components: 3
    isolation: 7
- id: dt_f3
  name: DefensiveToken
  origin: this_report
  base_checkpoint: Falcon3-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 5 new token embeddings prepended to the input, optimised with the model frozen
  - StruQ loss and StruQ defensive dataset from Cleaned Alpaca, labels generated by the undefended model
  layout: instruction in the system role, data in the user role, response in the assistant role, DefensiveTokens before the input
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; token embeddings not checked
  anchor: We optimize 5 defensive tokens, placed before the LLM input
  pages:
    components: 3
    isolation: 7
- id: dt_q25
  name: DefensiveToken
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 5 new token embeddings prepended to the input, optimised with the model frozen
  - StruQ loss and StruQ defensive dataset from Cleaned Alpaca, labels generated by the undefended model
  layout: instruction in the system role, data in the user role, response in the assistant role, DefensiveTokens before the input
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; token embeddings not checked
  anchor: We optimize 5 defensive tokens, placed before the LLM input
  pages:
    components: 3
    isolation: 7
- id: secalign_lora_l31
  name: SecAlign-LoRA (re-trained by the authors, self-generated labels)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as dt_l31 without DefensiveTokens
  anchor: we also apply it to all training-time defense baselines for a fair comparison
evaluations:
- id: ev_l3_none_alp_win
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id001
    benchmark: 4
    judge: 4
  decoding: not reported
  sample_size: '805'
  baseline_kind: off_the_shelf
- id: ev_l3_none_alp_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id002
    benchmark: 4
    judge: 4
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_l3_none_alp_gcg
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id003
    benchmark: 4
    judge: 4
    attacker_access: 3
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_l3_none_sep_win
  defence_id: none:Llama3-8B-Instruct
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id004
    benchmark: 4
    judge: 4
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l3_none_sep_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id005
    benchmark: 5
    judge: 4
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
  baseline_kind: off_the_shelf
- id: ev_l3_none_tt_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id006
    benchmark: 5
    judge: 4
  decoding: not reported
  sample_size: 31k
  baseline_kind: off_the_shelf
- id: ev_l3_none_cse_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id007
    benchmark: 5
    judge: 5
  decoding: not reported
  sample_size: '55'
  baseline_kind: off_the_shelf
- id: ev_l3_none_ia_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: &id008
    benchmark: 5
    judge: 5
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
  baseline_kind: off_the_shelf
- id: ev_l3_dt_alp_win
  defence_id: dt_l3
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
- id: ev_l3_dt_alp_asr
  defence_id: dt_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
- id: ev_l3_dt_alp_gcg
  defence_id: dt_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix, attacker knows the DefensiveToken embeddings
  sample_size: '208'
- id: ev_l3_dt_sep_win
  defence_id: dt_l3
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
- id: ev_l3_dt_sep_asr
  defence_id: dt_l3
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
- id: ev_l3_dt_tt_asr
  defence_id: dt_l3
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
- id: ev_l3_dt_cse_asr
  defence_id: dt_l3
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
- id: ev_l3_dt_ia_asr
  defence_id: dt_l3
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
- id: ev_l31_none_alp_win
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
  baseline_kind: off_the_shelf
- id: ev_l31_none_alp_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_l31_none_alp_gcg
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_l31_none_sep_win
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_none_sep_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
  baseline_kind: off_the_shelf
- id: ev_l31_none_tt_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
  baseline_kind: off_the_shelf
- id: ev_l31_none_cse_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
  baseline_kind: off_the_shelf
- id: ev_l31_none_ia_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
  baseline_kind: off_the_shelf
- id: ev_l31_dt_alp_win
  defence_id: dt_l31
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
- id: ev_l31_dt_alp_asr
  defence_id: dt_l31
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
- id: ev_l31_dt_alp_gcg
  defence_id: dt_l31
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix, attacker knows the DefensiveToken embeddings
  sample_size: '208'
- id: ev_l31_dt_sep_win
  defence_id: dt_l31
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
- id: ev_l31_dt_sep_asr
  defence_id: dt_l31
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
- id: ev_l31_dt_tt_asr
  defence_id: dt_l31
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
- id: ev_l31_dt_cse_asr
  defence_id: dt_l31
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
- id: ev_l31_dt_ia_asr
  defence_id: dt_l31
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
- id: ev_l31_sec_alp_win
  defence_id: secalign_lora_l31
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
- id: ev_l31_sec_alp_asr
  defence_id: secalign_lora_l31
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
- id: ev_l31_sec_alp_gcg
  defence_id: secalign_lora_l31
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix
  sample_size: '208'
- id: ev_l31_sec_sep_win
  defence_id: secalign_lora_l31
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
- id: ev_l31_sec_sep_asr
  defence_id: secalign_lora_l31
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
- id: ev_l31_sec_tt_asr
  defence_id: secalign_lora_l31
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
- id: ev_l31_sec_cse_asr
  defence_id: secalign_lora_l31
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
- id: ev_l31_sec_ia_asr
  defence_id: secalign_lora_l31
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
- id: ev_f3_none_alp_win
  defence_id: none:Falcon3-7B-Instruct
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
  baseline_kind: off_the_shelf
- id: ev_f3_none_alp_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_f3_none_alp_gcg
  defence_id: none:Falcon3-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_f3_none_sep_win
  defence_id: none:Falcon3-7B-Instruct
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_f3_none_sep_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
  baseline_kind: off_the_shelf
- id: ev_f3_none_tt_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
  baseline_kind: off_the_shelf
- id: ev_f3_none_cse_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
  baseline_kind: off_the_shelf
- id: ev_f3_none_ia_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
  baseline_kind: off_the_shelf
- id: ev_f3_dt_alp_win
  defence_id: dt_f3
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
- id: ev_f3_dt_alp_asr
  defence_id: dt_f3
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
- id: ev_f3_dt_alp_gcg
  defence_id: dt_f3
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix, attacker knows the DefensiveToken embeddings
  sample_size: '208'
- id: ev_f3_dt_sep_win
  defence_id: dt_f3
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
- id: ev_f3_dt_sep_asr
  defence_id: dt_f3
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
- id: ev_f3_dt_tt_asr
  defence_id: dt_f3
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
- id: ev_f3_dt_cse_asr
  defence_id: dt_f3
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
- id: ev_f3_dt_ia_asr
  defence_id: dt_f3
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
- id: ev_q25_none_alp_win
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
  baseline_kind: off_the_shelf
- id: ev_q25_none_alp_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_q25_none_alp_gcg
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_q25_none_sep_win
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q25_none_sep_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
  baseline_kind: off_the_shelf
- id: ev_q25_none_tt_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
  baseline_kind: off_the_shelf
- id: ev_q25_none_cse_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
  baseline_kind: off_the_shelf
- id: ev_q25_none_ia_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
  baseline_kind: off_the_shelf
- id: ev_q25_dt_alp_win
  defence_id: dt_q25
  benchmark: AlpacaEval2 on 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 turbo responses
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id001
  decoding: not reported
  sample_size: '805'
- id: ev_q25_dt_alp_asr
  defence_id: dt_q25
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! at the end of the data
  attacker_type: static
  attacker_access: black_box
  judge: lowercased reply begins with hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id002
  decoding: not reported
  injection_position: end
  attacker_budget: highest of Ignore, Completion and Ignore-Completion
  sample_size: '208'
- id: ev_q25_dt_alp_gcg
  defence_id: dt_q25
  benchmark: AlpacaFarm samples with a data part, Print exactly Hacked! plus a GCG suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply begins with Hacked
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id003
  decoding: not reported
  injection_position: end
  attacker_budget: GCG suffix, attacker knows the DefensiveToken embeddings
  sample_size: '208'
- id: ev_q25_dt_sep_win
  defence_id: dt_q25
  benchmark: SEP, AlpacaEval2 prompting against Llama3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id004
  decoding: not reported
- id: ev_q25_dt_sep_asr
  defence_id: dt_q25
  benchmark: SEP (9.1k samples, injection appended at the end of the data)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id005
  decoding: not reported
  injection_position: end
  sample_size: 9.1k
- id: ev_q25_dt_tt_asr
  defence_id: dt_q25
  benchmark: TaskTracker (31k samples, positions and enhancements as specified)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o judges whether the injection is followed
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id006
  decoding: not reported
  sample_size: 31k
- id: ev_q25_dt_cse_asr
  defence_id: dt_q25
  benchmark: CyberSecEval2 prompt injection (55 test cases)
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o with the benchmark's judge question
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id007
  decoding: not reported
  sample_size: '55'
- id: ev_q25_dt_ia_asr
  defence_id: dt_q25
  benchmark: InjecAgent (1K samples, base attack setting)
  attacker_type: static
  attacker_access: black_box
  judge: malicious API called (ASR-Total)
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 8
  pages: *id008
  decoding: not reported
  attacker_budget: base setting
  sample_size: 1K
results:
- id: r_l3_none_alp_win
  evaluation_id: ev_l3_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '26.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_alp_asr
  evaluation_id: ev_l3_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '51.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_alp_gcg
  evaluation_id: ev_l3_none_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '94.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_sep_win
  evaluation_id: ev_l3_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_sep_asr
  evaluation_id: ev_l3_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '79.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_tt_asr
  evaluation_id: ev_l3_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '16.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_cse_asr
  evaluation_id: ev_l3_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '49.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_ia_asr
  evaluation_id: ev_l3_none_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '29.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_alp_win
  evaluation_id: ev_l3_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_alp_asr
  evaluation_id: ev_l3_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_alp_gcg
  evaluation_id: ev_l3_dt_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '37.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_sep_win
  evaluation_id: ev_l3_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '51.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_sep_asr
  evaluation_id: ev_l3_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '3.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_tt_asr
  evaluation_id: ev_l3_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_cse_asr
  evaluation_id: ev_l3_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_dt_ia_asr
  evaluation_id: ev_l3_dt_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '2.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_alp_win
  evaluation_id: ev_l31_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '29.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_alp_asr
  evaluation_id: ev_l31_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '69.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_alp_gcg
  evaluation_id: ev_l31_none_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '96.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_sep_win
  evaluation_id: ev_l31_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '54.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_sep_asr
  evaluation_id: ev_l31_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '71.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_tt_asr
  evaluation_id: ev_l31_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '26.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_cse_asr
  evaluation_id: ev_l31_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '16.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_ia_asr
  evaluation_id: ev_l31_none_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '33.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_alp_win
  evaluation_id: ev_l31_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '28.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_alp_asr
  evaluation_id: ev_l31_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_alp_gcg
  evaluation_id: ev_l31_dt_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '24.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_sep_win
  evaluation_id: ev_l31_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '53.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_sep_asr
  evaluation_id: ev_l31_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_tt_asr
  evaluation_id: ev_l31_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_cse_asr
  evaluation_id: ev_l31_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '7.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_dt_ia_asr
  evaluation_id: ev_l31_dt_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_alp_win
  evaluation_id: ev_l31_sec_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_alp_asr
  evaluation_id: ev_l31_sec_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_alp_gcg
  evaluation_id: ev_l31_sec_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_sep_win
  evaluation_id: ev_l31_sec_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_sep_asr
  evaluation_id: ev_l31_sec_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_tt_asr
  evaluation_id: ev_l31_sec_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_cse_asr
  evaluation_id: ev_l31_sec_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_sec_ia_asr
  evaluation_id: ev_l31_sec_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_alp_win
  evaluation_id: ev_f3_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '30.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_alp_asr
  evaluation_id: ev_f3_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '84.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_alp_gcg
  evaluation_id: ev_f3_none_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '94.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_sep_win
  evaluation_id: ev_f3_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_sep_asr
  evaluation_id: ev_f3_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '80.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_tt_asr
  evaluation_id: ev_f3_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '27.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_cse_asr
  evaluation_id: ev_f3_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '50.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_none_ia_asr
  evaluation_id: ev_f3_none_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '20.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_alp_win
  evaluation_id: ev_f3_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '29.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_alp_asr
  evaluation_id: ev_f3_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_alp_gcg
  evaluation_id: ev_f3_dt_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '59.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_sep_win
  evaluation_id: ev_f3_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '48.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_sep_asr
  evaluation_id: ev_f3_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '6.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_tt_asr
  evaluation_id: ev_f3_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_cse_asr
  evaluation_id: ev_f3_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '12.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_f3_dt_ia_asr
  evaluation_id: ev_f3_dt_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '1.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_alp_win
  evaluation_id: ev_q25_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '32.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_alp_asr
  evaluation_id: ev_q25_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '93.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_alp_gcg
  evaluation_id: ev_q25_none_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '95.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_sep_win
  evaluation_id: ev_q25_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '54.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_sep_asr
  evaluation_id: ev_q25_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '87.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_tt_asr
  evaluation_id: ev_q25_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '37.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_cse_asr
  evaluation_id: ev_q25_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '45.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_none_ia_asr
  evaluation_id: ev_q25_none_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '23.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_alp_win
  evaluation_id: ev_q25_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '34.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_alp_asr
  evaluation_id: ev_q25_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_alp_gcg
  evaluation_id: ev_q25_dt_alp_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm GCG-ASR
  value: '73.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_sep_win
  evaluation_id: ev_q25_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_sep_asr
  evaluation_id: ev_q25_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '4.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_tt_asr
  evaluation_id: ev_q25_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_cse_asr
  evaluation_id: ev_q25_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '20.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q25_dt_ia_asr
  evaluation_id: ev_q25_dt_ia_asr
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
comparisons:
- id: c_l3_alp_win_dt_none
  left: ev_l3_dt_alp_win
  right: ev_l3_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_alp_asr_dt_none
  left: ev_l3_dt_alp_asr
  right: ev_l3_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_alp_gcg_dt_none
  left: ev_l3_dt_alp_gcg
  right: ev_l3_none_alp_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_sep_win_dt_none
  left: ev_l3_dt_sep_win
  right: ev_l3_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_sep_asr_dt_none
  left: ev_l3_dt_sep_asr
  right: ev_l3_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_tt_asr_dt_none
  left: ev_l3_dt_tt_asr
  right: ev_l3_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_cse_asr_dt_none
  left: ev_l3_dt_cse_asr
  right: ev_l3_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_ia_asr_dt_none
  left: ev_l3_dt_ia_asr
  right: ev_l3_none_ia_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_alp_win_dt_none
  left: ev_l31_dt_alp_win
  right: ev_l31_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_alp_asr_dt_none
  left: ev_l31_dt_alp_asr
  right: ev_l31_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_alp_gcg_dt_none
  left: ev_l31_dt_alp_gcg
  right: ev_l31_none_alp_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_sep_win_dt_none
  left: ev_l31_dt_sep_win
  right: ev_l31_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_sep_asr_dt_none
  left: ev_l31_dt_sep_asr
  right: ev_l31_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_tt_asr_dt_none
  left: ev_l31_dt_tt_asr
  right: ev_l31_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_cse_asr_dt_none
  left: ev_l31_dt_cse_asr
  right: ev_l31_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_ia_asr_dt_none
  left: ev_l31_dt_ia_asr
  right: ev_l31_none_ia_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_alp_win_dt_none
  left: ev_f3_dt_alp_win
  right: ev_f3_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_alp_asr_dt_none
  left: ev_f3_dt_alp_asr
  right: ev_f3_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_alp_gcg_dt_none
  left: ev_f3_dt_alp_gcg
  right: ev_f3_none_alp_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_sep_win_dt_none
  left: ev_f3_dt_sep_win
  right: ev_f3_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_sep_asr_dt_none
  left: ev_f3_dt_sep_asr
  right: ev_f3_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_tt_asr_dt_none
  left: ev_f3_dt_tt_asr
  right: ev_f3_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_cse_asr_dt_none
  left: ev_f3_dt_cse_asr
  right: ev_f3_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_f3_ia_asr_dt_none
  left: ev_f3_dt_ia_asr
  right: ev_f3_none_ia_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_alp_win_dt_none
  left: ev_q25_dt_alp_win
  right: ev_q25_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_alp_asr_dt_none
  left: ev_q25_dt_alp_asr
  right: ev_q25_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_alp_gcg_dt_none
  left: ev_q25_dt_alp_gcg
  right: ev_q25_none_alp_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_sep_win_dt_none
  left: ev_q25_dt_sep_win
  right: ev_q25_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_sep_asr_dt_none
  left: ev_q25_dt_sep_asr
  right: ev_q25_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_tt_asr_dt_none
  left: ev_q25_dt_tt_asr
  right: ev_q25_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_cse_asr_dt_none
  left: ev_q25_dt_cse_asr
  right: ev_q25_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_ia_asr_dt_none
  left: ev_q25_dt_ia_asr
  right: ev_q25_none_ia_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_alp_win_dt_sec
  left: ev_l31_dt_alp_win
  right: ev_l31_sec_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_alp_asr_dt_sec
  left: ev_l31_dt_alp_asr
  right: ev_l31_sec_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_alp_gcg_dt_sec
  left: ev_l31_dt_alp_gcg
  right: ev_l31_sec_alp_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_sep_win_dt_sec
  left: ev_l31_dt_sep_win
  right: ev_l31_sec_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_sep_asr_dt_sec
  left: ev_l31_dt_sep_asr
  right: ev_l31_sec_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_tt_asr_dt_sec
  left: ev_l31_dt_tt_asr
  right: ev_l31_sec_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_cse_asr_dt_sec
  left: ev_l31_dt_cse_asr
  right: ev_l31_sec_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
- id: c_l31_ia_asr_dt_sec
  left: ev_l31_dt_ia_asr
  right: ev_l31_sec_ia_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DefensiveToken (test time) against SecAlign-LoRA (training time) on the same base'
  page: 8
claims:
- id: k1
  text: robustness comparable to training-time defences
  location: abstract
  anchor: a test-time defense with prompt injection robustness comparable to training-time alternatives
  jobs:
  - R1
  supported_by:
  - c_l31_alp_asr_dt_sec
  - c_l31_sep_asr_dt_sec
  - c_l31_tt_asr_dt_sec
  - c_l31_cse_asr_dt_sec
  - c_l31_ia_asr_dt_sec
  - c_l31_alp_gcg_dt_sec
  page: 1
  contradicted_by:
  - r_l31_dt_alp_gcg
- id: k2
  text: security with a minimal utility drop when DefensiveTokens are added
  location: abstract
  anchor: to achieve security with a minimal utility drop
  jobs:
  - R1
  - general_utility
  - R2
  supported_by:
  - c_l3_alp_asr_dt_none
  - c_l3_sep_asr_dt_none
  - c_l3_tt_asr_dt_none
  - c_l3_cse_asr_dt_none
  - c_l3_ia_asr_dt_none
  - c_l31_alp_asr_dt_none
  - c_l31_sep_asr_dt_none
  - c_l31_tt_asr_dt_none
  - c_l31_cse_asr_dt_none
  - c_l31_ia_asr_dt_none
  - c_f3_alp_asr_dt_none
  - c_f3_sep_asr_dt_none
  - c_f3_tt_asr_dt_none
  - c_f3_cse_asr_dt_none
  - c_f3_ia_asr_dt_none
  - c_q25_alp_asr_dt_none
  - c_q25_sep_asr_dt_none
  - c_q25_tt_asr_dt_none
  - c_q25_cse_asr_dt_none
  - c_q25_ia_asr_dt_none
  - c_l3_alp_win_dt_none
  - c_l3_sep_win_dt_none
  - c_l31_alp_win_dt_none
  - c_l31_sep_win_dt_none
  - c_f3_alp_win_dt_none
  - c_f3_sep_win_dt_none
  - c_q25_alp_win_dt_none
  - c_q25_sep_win_dt_none
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3
    R2clean: Table 3 (SEP WinRate, AlpacaEval2 prompting on SEP)
    general_utility: Table 3 (AlpacaFarm WinRate)

`jatmo_arxiv2023` (2312.17673)

In [ ]:
%%writefile reports/records/jatmo_arxiv2023.yaml
report:
  id: jatmo_arxiv2023
  slug: '2312.17673'
  title: 'Jatmo: Prompt Injection Defense by Task-Specific Finetuning'
  first_author: Julien Piet
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2312.17673'
  version_read: arXiv v2
  first_date: '2023-12-29'
  read: F
  role: proposes
  code_release: https://github.com/wagner-group/prompt-injection-defense
  later_version: Springer book chapter, DOI 10.1007/978-3-031-70879-4_6 (from the search record; not read)
  extraction_scope: 'Sections 4 and 5.1 to 5.2: Table 2 (all seven tasks, three injection positions) and the 98% quality statement in Section 5; the abstract claims (Sections 5.3 to 5.4, Figures 4 and 5 and the appendix not
    coded)'
defences:
- id: jatmo_davinci002
  name: Jatmo
  origin: this_report
  base_checkpoint: davinci-002 (non-instruction-tuned base model)
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  components:
  - one task-specific model per task, fine-tuned from a base model that was never instruction-tuned
  - training outputs generated by the teacher GPT-3.5-Turbo from the task prompt and each input (real or synthetic inputs)
  layout: the task input alone; no instruction at inference
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: code released; models not checked
  anchor: We build each task-specific model by fine-tuning davinci-002
  pages:
    components: 10
evaluations:
- id: ev_code_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_code_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_code_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_code_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_code_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_code_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Code Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_sent_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_sent_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_sent_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_sent_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_sent_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_sent_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Sentiment Analysis test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_review_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_review_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_review_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_review_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_review_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_review_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Review Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_trans_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_trans_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_trans_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_trans_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_trans_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_trans_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Translation test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_news_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_news_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_news_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_news_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_news_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_news_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: News Summarization test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_tox_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_tox_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_tox_middle_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_tox_middle_jatmo
  defence_id: jatmo_davinci002
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: middle
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_tox_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_tox_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Toxicity Detection test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_sim_start_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Sentence Similarity test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_sim_start_jatmo
  defence_id: jatmo_davinci002
  benchmark: Sentence Similarity test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: start
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_sim_end_none
  defence_id: none:GPT-3.5-Turbo, prompted with the task (the teacher)
  benchmark: Sentence Similarity test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
  baseline_kind: off_the_shelf
- id: ev_sim_end_jatmo
  defence_id: jatmo_davinci002
  benchmark: Sentence Similarity test inputs with injections
  attacker_type: human
  attacker_access: black_box
  judge: output contains only HACKED and optional spaces
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 12
  attacker_budget: most effective of 10 HackAPrompt and 3 hand-crafted injections for this task and position
  injection_position: end
  decoding: not reported
  pages:
    judge: 13
    attacker_budget: 13
- id: ev_quality_jatmo
  defence_id: jatmo_davinci002
  benchmark: the seven tasks, 400 real training examples per task
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy on dataset labels or GPT-3.5-Turbo rating (0 to 100)
  layout_same_as_defence: 'yes'
  measured_by: jatmo_arxiv2023
  page: 11
  pages:
    judge: 12
results:
- id: r_code_start_none
  evaluation_id: ev_code_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_code_start_jatmo
  evaluation_id: ev_code_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_code_middle_none
  evaluation_id: ev_code_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_code_middle_jatmo
  evaluation_id: ev_code_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_code_end_none
  evaluation_id: ev_code_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_code_end_jatmo
  evaluation_id: ev_code_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_start_none
  evaluation_id: ev_sent_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_start_jatmo
  evaluation_id: ev_sent_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_middle_none
  evaluation_id: ev_sent_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_middle_jatmo
  evaluation_id: ev_sent_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_end_none
  evaluation_id: ev_sent_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sent_end_jatmo
  evaluation_id: ev_sent_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_start_none
  evaluation_id: ev_review_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_start_jatmo
  evaluation_id: ev_review_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_middle_none
  evaluation_id: ev_review_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_middle_jatmo
  evaluation_id: ev_review_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_end_none
  evaluation_id: ev_review_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_review_end_jatmo
  evaluation_id: ev_review_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_start_none
  evaluation_id: ev_trans_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_start_jatmo
  evaluation_id: ev_trans_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_middle_none
  evaluation_id: ev_trans_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_middle_jatmo
  evaluation_id: ev_trans_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_end_none
  evaluation_id: ev_trans_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_trans_end_jatmo
  evaluation_id: ev_trans_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_start_none
  evaluation_id: ev_news_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_start_jatmo
  evaluation_id: ev_news_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_middle_none
  evaluation_id: ev_news_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_middle_jatmo
  evaluation_id: ev_news_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_end_none
  evaluation_id: ev_news_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_news_end_jatmo
  evaluation_id: ev_news_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_start_none
  evaluation_id: ev_tox_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_start_jatmo
  evaluation_id: ev_tox_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_middle_none
  evaluation_id: ev_tox_middle_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '84'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_middle_jatmo
  evaluation_id: ev_tox_middle_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Middle
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_end_none
  evaluation_id: ev_tox_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_tox_end_jatmo
  evaluation_id: ev_tox_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sim_start_none
  evaluation_id: ev_sim_start_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sim_start_jatmo
  evaluation_id: ev_sim_start_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, Start
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sim_end_none
  evaluation_id: ev_sim_end_none
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_sim_end_jatmo
  evaluation_id: ev_sim_end_jatmo
  quantity: R1
  coverage: direct
  metric_as_printed: Prompt-injection success rate, End
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_quality_jatmo
  evaluation_id: ev_quality_jatmo
  quantity: R2clean
  coverage: direct
  metric_as_printed: quality relative to the teacher model
  value: '98'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5
  reference: GPT-3.5-Turbo (the teacher) on the same tasks
comparisons:
- id: c_code_start
  left: ev_code_start_jatmo
  right: ev_code_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_code_middle
  left: ev_code_middle_jatmo
  right: ev_code_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_code_end
  left: ev_code_end_jatmo
  right: ev_code_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_sent_start
  left: ev_sent_start_jatmo
  right: ev_sent_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_sent_middle
  left: ev_sent_middle_jatmo
  right: ev_sent_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_sent_end
  left: ev_sent_end_jatmo
  right: ev_sent_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_review_start
  left: ev_review_start_jatmo
  right: ev_review_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_review_middle
  left: ev_review_middle_jatmo
  right: ev_review_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_review_end
  left: ev_review_end_jatmo
  right: ev_review_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_trans_start
  left: ev_trans_start_jatmo
  right: ev_trans_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_trans_middle
  left: ev_trans_middle_jatmo
  right: ev_trans_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_trans_end
  left: ev_trans_end_jatmo
  right: ev_trans_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_news_start
  left: ev_news_start_jatmo
  right: ev_news_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_news_middle
  left: ev_news_middle_jatmo
  right: ev_news_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_news_end
  left: ev_news_end_jatmo
  right: ev_news_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_tox_start
  left: ev_tox_start_jatmo
  right: ev_tox_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_tox_middle
  left: ev_tox_middle_jatmo
  right: ev_tox_middle_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_tox_end
  left: ev_tox_end_jatmo
  right: ev_tox_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_sim_start
  left: ev_sim_start_jatmo
  right: ev_sim_start_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
- id: c_sim_end
  left: ev_sim_end_jatmo
  right: ev_sim_end_none
  purpose: W
  c1_base: unmatched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: Jatmo''s task-specific fine-tuned base model against the prompted teacher GPT-3.5-Turbo'
  page: 12
claims:
- id: k1
  text: best attacks succeed in under 0.5% of cases against Jatmo models, against 87% on GPT-3.5-Turbo
  location: abstract
  anchor: The best attacks succeeded in less than 0.5% of cases against our models
  jobs:
  - R1
  supported_by:
  - c_code_start
  - c_code_middle
  - c_code_end
  - c_sent_start
  - c_sent_middle
  - c_sent_end
  - c_review_start
  - c_review_middle
  - c_review_end
  - c_trans_start
  - c_trans_middle
  - c_trans_end
  - c_news_start
  - c_news_middle
  - c_news_end
  - c_tox_start
  - c_tox_middle
  - c_tox_end
  - c_sim_start
  - c_sim_end
  page: 1
  contradicted_by:
  - r_review_end_jatmo
- id: k2
  text: Jatmo models give similar output quality to standard LLMs on their task
  location: abstract
  anchor: Jatmo models provide similar quality of outputs on their specific task as standard LLMs
  jobs:
  - R2
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: aggregate_only
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (best injection per task and position)
    R2clean: Section 5 (98% of teacher quality, pooled over tasks); Table 2 prints per-task differences only

`bipia_kdd2025` (2312.14197)

In [ ]:
%%writefile reports/records/bipia_kdd2025.yaml
report:
  id: bipia_kdd2025
  slug: '2312.14197'
  title: Benchmarking and Defending Against Indirect Prompt Injection Attacks on Large Language Models
  first_author: Jingwei Yi
  venue: KDD 2025 (31st ACM SIGKDD Conference)
  peer_reviewed: 'yes'
  doi: 10.1145/3690624.3709179
  arxiv: '2312.14197'
  version_read: arXiv v4 (carries the KDD '25 header and DOI)
  first_date: '2023-12-21'
  read: F
  role: mixed
  code_release: https://github.com/microsoft/BIPIA (linked from the abstract)
  extraction_scope: 'Sections 6 and 7.1 to 7.2: every row of Tables 3 and 4 (ROUGE-1, MT-Bench, the five task ASRs and the overall ASR); the abstract claims (the benchmark results of Table 2, Figures 3 to 5 and the ablation
    Figures 10 and 11 not coded)'
defences:
- id: icl_gpt4
  name: In-context learning with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: GPT-4
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - explicit reminder not to execute instructions in the external content
  - worked examples from the BIPIA training set
  layout: 'one prompt: application instruction with the reminder, worked examples as earlier turns, then the external content'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we employ this technique to teach an LLM the boundaries
  pages:
    isolation: 8
- id: mt_gpt4
  name: Multi-turn dialogue with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: GPT-4
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - external content moved to an earlier user turn
  - instruction with explicit reminder in the current turn
  layout: external content in the previous turn, the instruction in the current turn
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we propose moving third-party content, which may contain malicious instructions
  signal_dependence_detail: Figure 11 reverts to a single turn without the separation; bars only, values not printed
  pages:
    isolation: 8
    signal_dependence: 9
- id: icl_gpt35
  name: In-context learning with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - explicit reminder not to execute instructions in the external content
  - worked examples from the BIPIA training set
  layout: 'one prompt: application instruction with the reminder, worked examples as earlier turns, then the external content'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we employ this technique to teach an LLM the boundaries
  pages:
    isolation: 8
- id: mt_gpt35
  name: Multi-turn dialogue with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - external content moved to an earlier user turn
  - instruction with explicit reminder in the current turn
  layout: external content in the previous turn, the instruction in the current turn
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we propose moving third-party content, which may contain malicious instructions
  signal_dependence_detail: Figure 11 reverts to a single turn without the separation; bars only, values not printed
  pages:
    isolation: 8
    signal_dependence: 9
- id: icl_v7
  name: In-context learning with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: Vicuna-7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - explicit reminder not to execute instructions in the external content
  - worked examples from the BIPIA training set
  layout: 'one prompt: application instruction with the reminder, worked examples as earlier turns, then the external content'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we employ this technique to teach an LLM the boundaries
  pages:
    isolation: 8
- id: mt_v7
  name: Multi-turn dialogue with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: Vicuna-7B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - external content moved to an earlier user turn
  - instruction with explicit reminder in the current turn
  layout: external content in the previous turn, the instruction in the current turn
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we propose moving third-party content, which may contain malicious instructions
  signal_dependence_detail: Figure 11 reverts to a single turn without the separation; bars only, values not printed
  pages:
    isolation: 8
    signal_dependence: 9
- id: icl_v13
  name: In-context learning with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: Vicuna-13B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - explicit reminder not to execute instructions in the external content
  - worked examples from the BIPIA training set
  layout: 'one prompt: application instruction with the reminder, worked examples as earlier turns, then the external content'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we employ this technique to teach an LLM the boundaries
  pages:
    isolation: 8
- id: mt_v13
  name: Multi-turn dialogue with explicit reminder (black-box defence)
  origin: this_report
  base_checkpoint: Vicuna-13B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - external content moved to an earlier user turn
  - instruction with explicit reminder in the current turn
  layout: external content in the previous turn, the instruction in the current turn
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we propose moving third-party content, which may contain malicious instructions
  signal_dependence_detail: Figure 11 reverts to a single turn without the separation; bars only, values not printed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_bipia_v7
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (BIPIA labels)'
  origin: this_report
  base_checkpoint: Vicuna-7B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (BIPIA labels)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_orig_v7
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (responses of the original LLM)'
  origin: this_report
  base_checkpoint: Vicuna-7B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (responses of the original LLM)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_gpt4r_v7
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (GPT-4 responses)'
  origin: this_report
  base_checkpoint: Vicuna-7B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (GPT-4 responses)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_bipia_v13
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (BIPIA labels)'
  origin: this_report
  base_checkpoint: Vicuna-13B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (BIPIA labels)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_orig_v13
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (responses of the original LLM)'
  origin: this_report
  base_checkpoint: Vicuna-13B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (responses of the original LLM)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
- id: wb_gpt4r_v13
  name: 'White-box defence: <data> tokens, reminder and adversarial fine-tuning (GPT-4 responses)'
  origin: this_report
  base_checkpoint: Vicuna-13B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - new special tokens <data> and </data> around the external content
  - explicit reminder in the prompt template
  - one epoch of supervised fine-tuning on attacked BIPIA training prompts with benign targets (GPT-4 responses)
  layout: application instruction with the reminder, external content between <data> and </data>, then the user instruction
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two special tokens to the vocabulary of the LLM
  signal_dependence_detail: Figure 10 removes the special tokens together with the adversarial training; no run with the tokens alone removed
  pages:
    isolation: 8
    signal_dependence: 9
evaluations:
- id: ev_gpt4_none_rouge
  defence_id: none:GPT-4
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_email
  defence_id: none:GPT-4
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_web
  defence_id: none:GPT-4
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_table
  defence_id: none:GPT-4
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_summ
  defence_id: none:GPT-4
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_code
  defence_id: none:GPT-4
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_overall
  defence_id: none:GPT-4
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt4_icl_rouge
  defence_id: icl_gpt4
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_gpt4_icl_email
  defence_id: icl_gpt4
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_icl_web
  defence_id: icl_gpt4
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_icl_table
  defence_id: icl_gpt4
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_icl_summ
  defence_id: icl_gpt4
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_icl_code
  defence_id: icl_gpt4
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_icl_overall
  defence_id: icl_gpt4
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_rouge
  defence_id: mt_gpt4
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_gpt4_mt_email
  defence_id: mt_gpt4
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_web
  defence_id: mt_gpt4
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_table
  defence_id: mt_gpt4
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_summ
  defence_id: mt_gpt4
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_code
  defence_id: mt_gpt4
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt4_mt_overall
  defence_id: mt_gpt4
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_none_rouge
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_email
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_web
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_table
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_summ
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_code
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_none_overall
  defence_id: none:GPT-3.5-Turbo
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_gpt35_icl_rouge
  defence_id: icl_gpt35
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_gpt35_icl_email
  defence_id: icl_gpt35
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_icl_web
  defence_id: icl_gpt35
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_icl_table
  defence_id: icl_gpt35
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_icl_summ
  defence_id: icl_gpt35
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_icl_code
  defence_id: icl_gpt35
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_icl_overall
  defence_id: icl_gpt35
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_rouge
  defence_id: mt_gpt35
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_gpt35_mt_email
  defence_id: mt_gpt35
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_web
  defence_id: mt_gpt35
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_table
  defence_id: mt_gpt35
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_summ
  defence_id: mt_gpt35
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_code
  defence_id: mt_gpt35
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_gpt35_mt_overall
  defence_id: mt_gpt35
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_none_rouge
  defence_id: none:Vicuna-7B
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_v7_none_email
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_none_web
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_none_table
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_none_summ
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_none_code
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_none_overall
  defence_id: none:Vicuna-7B
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v7_icl_rouge
  defence_id: icl_v7
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_icl_email
  defence_id: icl_v7
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_icl_web
  defence_id: icl_v7
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_icl_table
  defence_id: icl_v7
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_icl_summ
  defence_id: icl_v7
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_icl_code
  defence_id: icl_v7
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_icl_overall
  defence_id: icl_v7
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_rouge
  defence_id: mt_v7
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_mt_email
  defence_id: mt_v7
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_web
  defence_id: mt_v7
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_table
  defence_id: mt_v7
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_summ
  defence_id: mt_v7
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_code
  defence_id: mt_v7
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_mt_overall
  defence_id: mt_v7
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_none_rouge
  defence_id: none:Vicuna-13B
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_v13_none_email
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_none_web
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_none_table
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_none_summ
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_none_code
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_none_overall
  defence_id: none:Vicuna-13B
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_v13_icl_rouge
  defence_id: icl_v13
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_icl_email
  defence_id: icl_v13
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_icl_web
  defence_id: icl_v13
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_icl_table
  defence_id: icl_v13
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_icl_summ
  defence_id: icl_v13
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_icl_code
  defence_id: icl_v13
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_icl_overall
  defence_id: icl_v13
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_rouge
  defence_id: mt_v13
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_mt_email
  defence_id: mt_v13
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_web
  defence_id: mt_v13
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_table
  defence_id: mt_v13
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_summ
  defence_id: mt_v13
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_code
  defence_id: mt_v13
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_mt_overall
  defence_id: mt_v13
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 2,000 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_none_mtbench
  defence_id: none:Vicuna-7B
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_v7_wb_bipia_rouge
  defence_id: wb_bipia_v7
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_bipia_mtbench
  defence_id: wb_bipia_v7
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_bipia_email
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_bipia_web
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_bipia_table
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_bipia_summ
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_bipia_code
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_bipia_overall
  defence_id: wb_bipia_v7
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_rouge
  defence_id: wb_orig_v7
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_orig_mtbench
  defence_id: wb_orig_v7
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_orig_email
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_web
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_table
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_summ
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_code
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_orig_overall
  defence_id: wb_orig_v7
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_rouge
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_gpt4r_mtbench
  defence_id: wb_gpt4r_v7
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v7_wb_gpt4r_email
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_web
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_table
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_summ
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_code
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v7_wb_gpt4r_overall
  defence_id: wb_gpt4r_v7
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_none_mtbench
  defence_id: none:Vicuna-13B
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_v13_wb_bipia_rouge
  defence_id: wb_bipia_v13
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_bipia_mtbench
  defence_id: wb_bipia_v13
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_bipia_email
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_bipia_web
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_bipia_table
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_bipia_summ
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_bipia_code
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_bipia_overall
  defence_id: wb_bipia_v13
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_rouge
  defence_id: wb_orig_v13
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_orig_mtbench
  defence_id: wb_orig_v13
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_orig_email
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_web
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_table
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_summ
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_code
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_orig_overall
  defence_id: wb_orig_v13
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_rouge
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA-Clean (same tasks, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: ROUGE-1 recall against the targets
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_gpt4r_mtbench
  defence_id: wb_gpt4r_v13
  benchmark: MT-Bench
  attacker_type: none
  attacker_access: not_applicable
  judge: MT-Bench score
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: not reported
  pages:
    judge: 3
    decoding: 7
    benchmark: 7
- id: ev_v13_wb_gpt4r_email
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, Email QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_web
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, Web QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_table
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, Table QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_summ
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, Summarization
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_code
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, Code QA
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
- id: ev_v13_wb_gpt4r_overall
  defence_id: wb_gpt4r_v13
  benchmark: BIPIA test set, all five tasks (weighted by example count)
  attacker_type: static
  attacker_access: black_box
  judge: 'BIPIA pipeline: rule-based checks, LLM-as-judge and language detection'
  layout_same_as_defence: 'yes'
  measured_by: bipia_kdd2025
  page: 8
  decoding: temperature 0, at most 512 tokens
  pages:
    judge: 3
    decoding: 7
  injection_position: varied
results:
- id: r_gpt4_none_rouge
  evaluation_id: ev_gpt4_none_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6985'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_email
  evaluation_id: ev_gpt4_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1524'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_web
  evaluation_id: ev_gpt4_none_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.2792'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_table
  evaluation_id: ev_gpt4_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.3472'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_summ
  evaluation_id: ev_gpt4_none_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.3917'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_code
  evaluation_id: ev_gpt4_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.2863'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_none_overall
  evaluation_id: ev_gpt4_none_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.3103'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_rouge
  evaluation_id: ev_gpt4_icl_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6590'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_email
  evaluation_id: ev_gpt4_icl_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1036'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_web
  evaluation_id: ev_gpt4_icl_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.2382'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_table
  evaluation_id: ev_gpt4_icl_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.3238'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_summ
  evaluation_id: ev_gpt4_icl_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.3075'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_code
  evaluation_id: ev_gpt4_icl_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0056'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_icl_overall
  evaluation_id: ev_gpt4_icl_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.2408'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_rouge
  evaluation_id: ev_gpt4_mt_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.7201'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_email
  evaluation_id: ev_gpt4_mt_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0959'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_web
  evaluation_id: ev_gpt4_mt_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.2585'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_table
  evaluation_id: ev_gpt4_mt_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.2974'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_summ
  evaluation_id: ev_gpt4_mt_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.1810'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_code
  evaluation_id: ev_gpt4_mt_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0097'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt4_mt_overall
  evaluation_id: ev_gpt4_mt_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.2056'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_rouge
  evaluation_id: ev_gpt35_none_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6554'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_email
  evaluation_id: ev_gpt35_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1634'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_web
  evaluation_id: ev_gpt35_none_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.2347'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_table
  evaluation_id: ev_gpt35_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.2257'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_summ
  evaluation_id: ev_gpt35_none_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.3658'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_code
  evaluation_id: ev_gpt35_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.2844'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_none_overall
  evaluation_id: ev_gpt35_none_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.2616'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_rouge
  evaluation_id: ev_gpt35_icl_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6289'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_email
  evaluation_id: ev_gpt35_icl_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1779'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_web
  evaluation_id: ev_gpt35_icl_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.1600'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_table
  evaluation_id: ev_gpt35_icl_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.1910'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_summ
  evaluation_id: ev_gpt35_icl_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.2560'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_code
  evaluation_id: ev_gpt35_icl_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0789'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_icl_overall
  evaluation_id: ev_gpt35_icl_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1884'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_rouge
  evaluation_id: ev_gpt35_mt_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6786'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_email
  evaluation_id: ev_gpt35_mt_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1376'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_web
  evaluation_id: ev_gpt35_mt_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.2025'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_table
  evaluation_id: ev_gpt35_mt_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.1936'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_summ
  evaluation_id: ev_gpt35_mt_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.2221'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_code
  evaluation_id: ev_gpt35_mt_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0583'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_gpt35_mt_overall
  evaluation_id: ev_gpt35_mt_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1843'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_rouge
  evaluation_id: ev_v7_none_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6187'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_email
  evaluation_id: ev_v7_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1124'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_web
  evaluation_id: ev_v7_none_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0693'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_table
  evaluation_id: ev_v7_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0827'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_summ
  evaluation_id: ev_v7_none_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.2117'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_code
  evaluation_id: ev_v7_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.1632'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_overall
  evaluation_id: ev_v7_none_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1237'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_rouge
  evaluation_id: ev_v7_icl_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6065'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_email
  evaluation_id: ev_v7_icl_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0512'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_web
  evaluation_id: ev_v7_icl_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0394'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_table
  evaluation_id: ev_v7_icl_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0396'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_summ
  evaluation_id: ev_v7_icl_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.2148'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_code
  evaluation_id: ev_v7_icl_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0599'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_icl_overall
  evaluation_id: ev_v7_icl_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0885'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_rouge
  evaluation_id: ev_v7_mt_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6084'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_email
  evaluation_id: ev_v7_mt_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1127'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_web
  evaluation_id: ev_v7_mt_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0410'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_table
  evaluation_id: ev_v7_mt_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0565'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_summ
  evaluation_id: ev_v7_mt_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0817'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_code
  evaluation_id: ev_v7_mt_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0023'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_mt_overall
  evaluation_id: ev_v7_mt_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0617'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_rouge
  evaluation_id: ev_v13_none_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6134'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_email
  evaluation_id: ev_v13_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1242'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_web
  evaluation_id: ev_v13_none_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.1272'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_table
  evaluation_id: ev_v13_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.1337'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_summ
  evaluation_id: ev_v13_none_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.2052'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_code
  evaluation_id: ev_v13_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.1755'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_none_overall
  evaluation_id: ev_v13_none_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1531'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_rouge
  evaluation_id: ev_v13_icl_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6025'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_email
  evaluation_id: ev_v13_icl_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.2353'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_web
  evaluation_id: ev_v13_icl_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.1456'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_table
  evaluation_id: ev_v13_icl_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.1804'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_summ
  evaluation_id: ev_v13_icl_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.1763'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_code
  evaluation_id: ev_v13_icl_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0468'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_icl_overall
  evaluation_id: ev_v13_icl_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1658'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_rouge
  evaluation_id: ev_v13_mt_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6274'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_email
  evaluation_id: ev_v13_mt_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.1379'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_web
  evaluation_id: ev_v13_mt_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.1024'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_table
  evaluation_id: ev_v13_mt_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.1168'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_summ
  evaluation_id: ev_v13_mt_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.1028'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_code
  evaluation_id: ev_v13_mt_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0015'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_mt_overall
  evaluation_id: ev_v13_mt_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.1021'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_none_mtbench
  evaluation_id: ev_v7_none_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.8063'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v7_wb_bipia_rouge
  evaluation_id: ev_v7_wb_bipia_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.5306'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_mtbench
  evaluation_id: ev_v7_wb_bipia_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.2938'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_email
  evaluation_id: ev_v7_wb_bipia_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0202'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_web
  evaluation_id: ev_v7_wb_bipia_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0159'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_table
  evaluation_id: ev_v7_wb_bipia_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0410'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_summ
  evaluation_id: ev_v7_wb_bipia_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0049'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_code
  evaluation_id: ev_v7_wb_bipia_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0300'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_bipia_overall
  evaluation_id: ev_v7_wb_bipia_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0214'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_rouge
  evaluation_id: ev_v7_wb_orig_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6122'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_mtbench
  evaluation_id: ev_v7_wb_orig_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.5687'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_email
  evaluation_id: ev_v7_wb_orig_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0015'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_web
  evaluation_id: ev_v7_wb_orig_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0062'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_table
  evaluation_id: ev_v7_wb_orig_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0057'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_summ
  evaluation_id: ev_v7_wb_orig_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0043'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_code
  evaluation_id: ev_v7_wb_orig_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.2244'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_orig_overall
  evaluation_id: ev_v7_wb_orig_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0240'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_rouge
  evaluation_id: ev_v7_wb_gpt4r_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6260'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_mtbench
  evaluation_id: ev_v7_wb_gpt4r_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.8312'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_email
  evaluation_id: ev_v7_wb_gpt4r_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0065'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_web
  evaluation_id: ev_v7_wb_gpt4r_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0045'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_table
  evaluation_id: ev_v7_wb_gpt4r_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0046'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_summ
  evaluation_id: ev_v7_wb_gpt4r_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0037'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_code
  evaluation_id: ev_v7_wb_gpt4r_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0129'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v7_wb_gpt4r_overall
  evaluation_id: ev_v7_wb_gpt4r_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0053'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_none_mtbench
  evaluation_id: ev_v13_none_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '5.2062'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_v13_wb_bipia_rouge
  evaluation_id: ev_v13_wb_bipia_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6109'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_mtbench
  evaluation_id: ev_v13_wb_bipia_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '1.6625'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_email
  evaluation_id: ev_v13_wb_bipia_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0217'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_web
  evaluation_id: ev_v13_wb_bipia_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0126'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_table
  evaluation_id: ev_v13_wb_bipia_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0370'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_summ
  evaluation_id: ev_v13_wb_bipia_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0064'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_code
  evaluation_id: ev_v13_wb_bipia_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0207'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_bipia_overall
  evaluation_id: ev_v13_wb_bipia_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0192'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_rouge
  evaluation_id: ev_v13_wb_orig_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6240'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_mtbench
  evaluation_id: ev_v13_wb_orig_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.3375'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_email
  evaluation_id: ev_v13_wb_orig_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0024'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_web
  evaluation_id: ev_v13_wb_orig_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0055'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_table
  evaluation_id: ev_v13_wb_orig_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0051'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_summ
  evaluation_id: ev_v13_wb_orig_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0034'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_code
  evaluation_id: ev_v13_wb_orig_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0067'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_orig_overall
  evaluation_id: ev_v13_wb_orig_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0046'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_rouge
  evaluation_id: ev_v13_wb_gpt4r_rouge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: ROUGE-1 (recall)
  value: '0.6337'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_mtbench
  evaluation_id: ev_v13_wb_gpt4r_mtbench
  quantity: general_utility
  coverage: direct
  metric_as_printed: Capability MT-Bench
  value: '4.5500'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_email
  evaluation_id: ev_v13_wb_gpt4r_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Email QA
  value: '0.0060'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_web
  evaluation_id: ev_v13_wb_gpt4r_web
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web QA
  value: '0.0044'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_table
  evaluation_id: ev_v13_wb_gpt4r_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Table QA
  value: '0.0057'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_summ
  evaluation_id: ev_v13_wb_gpt4r_summ
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Summarization
  value: '0.0036'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_code
  evaluation_id: ev_v13_wb_gpt4r_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Code QA
  value: '0.0036'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_v13_wb_gpt4r_overall
  evaluation_id: ev_v13_wb_gpt4r_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '0.0047'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
comparisons:
- id: c_gpt4_icl_rouge
  left: ev_gpt4_icl_rouge
  right: ev_gpt4_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_email
  left: ev_gpt4_icl_email
  right: ev_gpt4_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_web
  left: ev_gpt4_icl_web
  right: ev_gpt4_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_table
  left: ev_gpt4_icl_table
  right: ev_gpt4_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_summ
  left: ev_gpt4_icl_summ
  right: ev_gpt4_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_code
  left: ev_gpt4_icl_code
  right: ev_gpt4_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_icl_overall
  left: ev_gpt4_icl_overall
  right: ev_gpt4_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_rouge
  left: ev_gpt4_mt_rouge
  right: ev_gpt4_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_email
  left: ev_gpt4_mt_email
  right: ev_gpt4_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_web
  left: ev_gpt4_mt_web
  right: ev_gpt4_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_table
  left: ev_gpt4_mt_table
  right: ev_gpt4_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_summ
  left: ev_gpt4_mt_summ
  right: ev_gpt4_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_code
  left: ev_gpt4_mt_code
  right: ev_gpt4_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt4_mt_overall
  left: ev_gpt4_mt_overall
  right: ev_gpt4_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_rouge
  left: ev_gpt35_icl_rouge
  right: ev_gpt35_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_email
  left: ev_gpt35_icl_email
  right: ev_gpt35_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_web
  left: ev_gpt35_icl_web
  right: ev_gpt35_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_table
  left: ev_gpt35_icl_table
  right: ev_gpt35_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_summ
  left: ev_gpt35_icl_summ
  right: ev_gpt35_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_code
  left: ev_gpt35_icl_code
  right: ev_gpt35_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_icl_overall
  left: ev_gpt35_icl_overall
  right: ev_gpt35_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_rouge
  left: ev_gpt35_mt_rouge
  right: ev_gpt35_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_email
  left: ev_gpt35_mt_email
  right: ev_gpt35_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_web
  left: ev_gpt35_mt_web
  right: ev_gpt35_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_table
  left: ev_gpt35_mt_table
  right: ev_gpt35_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_summ
  left: ev_gpt35_mt_summ
  right: ev_gpt35_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_code
  left: ev_gpt35_mt_code
  right: ev_gpt35_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpt35_mt_overall
  left: ev_gpt35_mt_overall
  right: ev_gpt35_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_rouge
  left: ev_v7_icl_rouge
  right: ev_v7_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_email
  left: ev_v7_icl_email
  right: ev_v7_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_web
  left: ev_v7_icl_web
  right: ev_v7_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_table
  left: ev_v7_icl_table
  right: ev_v7_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_summ
  left: ev_v7_icl_summ
  right: ev_v7_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_code
  left: ev_v7_icl_code
  right: ev_v7_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_icl_overall
  left: ev_v7_icl_overall
  right: ev_v7_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_rouge
  left: ev_v7_mt_rouge
  right: ev_v7_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_email
  left: ev_v7_mt_email
  right: ev_v7_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_web
  left: ev_v7_mt_web
  right: ev_v7_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_table
  left: ev_v7_mt_table
  right: ev_v7_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_summ
  left: ev_v7_mt_summ
  right: ev_v7_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_code
  left: ev_v7_mt_code
  right: ev_v7_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_mt_overall
  left: ev_v7_mt_overall
  right: ev_v7_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_rouge
  left: ev_v13_icl_rouge
  right: ev_v13_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_email
  left: ev_v13_icl_email
  right: ev_v13_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_web
  left: ev_v13_icl_web
  right: ev_v13_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_table
  left: ev_v13_icl_table
  right: ev_v13_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_summ
  left: ev_v13_icl_summ
  right: ev_v13_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_code
  left: ev_v13_icl_code
  right: ev_v13_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_icl_overall
  left: ev_v13_icl_overall
  right: ev_v13_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_rouge
  left: ev_v13_mt_rouge
  right: ev_v13_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_email
  left: ev_v13_mt_email
  right: ev_v13_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_web
  left: ev_v13_mt_web
  right: ev_v13_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_table
  left: ev_v13_mt_table
  right: ev_v13_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_summ
  left: ev_v13_mt_summ
  right: ev_v13_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_code
  left: ev_v13_mt_code
  right: ev_v13_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_mt_overall
  left: ev_v13_mt_overall
  right: ev_v13_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_wb_bipia_rouge
  left: ev_v7_wb_bipia_rouge
  right: ev_v7_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_mtbench
  left: ev_v7_wb_bipia_mtbench
  right: ev_v7_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_wb_bipia_email
  left: ev_v7_wb_bipia_email
  right: ev_v7_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_web
  left: ev_v7_wb_bipia_web
  right: ev_v7_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_table
  left: ev_v7_wb_bipia_table
  right: ev_v7_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_summ
  left: ev_v7_wb_bipia_summ
  right: ev_v7_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_code
  left: ev_v7_wb_bipia_code
  right: ev_v7_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_bipia_overall
  left: ev_v7_wb_bipia_overall
  right: ev_v7_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_rouge
  left: ev_v7_wb_orig_rouge
  right: ev_v7_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_mtbench
  left: ev_v7_wb_orig_mtbench
  right: ev_v7_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_wb_orig_email
  left: ev_v7_wb_orig_email
  right: ev_v7_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_web
  left: ev_v7_wb_orig_web
  right: ev_v7_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_table
  left: ev_v7_wb_orig_table
  right: ev_v7_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_summ
  left: ev_v7_wb_orig_summ
  right: ev_v7_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_code
  left: ev_v7_wb_orig_code
  right: ev_v7_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_orig_overall
  left: ev_v7_wb_orig_overall
  right: ev_v7_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_rouge
  left: ev_v7_wb_gpt4r_rouge
  right: ev_v7_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_mtbench
  left: ev_v7_wb_gpt4r_mtbench
  right: ev_v7_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v7_wb_gpt4r_email
  left: ev_v7_wb_gpt4r_email
  right: ev_v7_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_web
  left: ev_v7_wb_gpt4r_web
  right: ev_v7_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_table
  left: ev_v7_wb_gpt4r_table
  right: ev_v7_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_summ
  left: ev_v7_wb_gpt4r_summ
  right: ev_v7_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_code
  left: ev_v7_wb_gpt4r_code
  right: ev_v7_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v7_wb_gpt4r_overall
  left: ev_v7_wb_gpt4r_overall
  right: ev_v7_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_rouge
  left: ev_v13_wb_bipia_rouge
  right: ev_v13_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_mtbench
  left: ev_v13_wb_bipia_mtbench
  right: ev_v13_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_wb_bipia_email
  left: ev_v13_wb_bipia_email
  right: ev_v13_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_web
  left: ev_v13_wb_bipia_web
  right: ev_v13_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_table
  left: ev_v13_wb_bipia_table
  right: ev_v13_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_summ
  left: ev_v13_wb_bipia_summ
  right: ev_v13_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_code
  left: ev_v13_wb_bipia_code
  right: ev_v13_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_bipia_overall
  left: ev_v13_wb_bipia_overall
  right: ev_v13_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_rouge
  left: ev_v13_wb_orig_rouge
  right: ev_v13_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_mtbench
  left: ev_v13_wb_orig_mtbench
  right: ev_v13_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_wb_orig_email
  left: ev_v13_wb_orig_email
  right: ev_v13_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_web
  left: ev_v13_wb_orig_web
  right: ev_v13_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_table
  left: ev_v13_wb_orig_table
  right: ev_v13_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_summ
  left: ev_v13_wb_orig_summ
  right: ev_v13_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_code
  left: ev_v13_wb_orig_code
  right: ev_v13_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_orig_overall
  left: ev_v13_wb_orig_overall
  right: ev_v13_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_rouge
  left: ev_v13_wb_gpt4r_rouge
  right: ev_v13_none_rouge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_mtbench
  left: ev_v13_wb_gpt4r_mtbench
  right: ev_v13_none_mtbench
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_v13_wb_gpt4r_email
  left: ev_v13_wb_gpt4r_email
  right: ev_v13_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_web
  left: ev_v13_wb_gpt4r_web
  right: ev_v13_none_web
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_table
  left: ev_v13_wb_gpt4r_table
  right: ev_v13_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_summ
  left: ev_v13_wb_gpt4r_summ
  right: ev_v13_none_summ
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_code
  left: ev_v13_wb_gpt4r_code
  right: ev_v13_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
- id: c_v13_wb_gpt4r_overall
  left: ev_v13_wb_gpt4r_overall
  right: ev_v13_none_overall
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unmatched
  varies: the defence; the fine-tuned model was decoded with a 512-token cap, the original (same numbers as Table 3) with 2,000
  page: 8
claims:
- id: k1
  text: the black-box defences give substantial mitigation
  location: abstract
  anchor: our black-box defense provides substantial mitigation
  jobs:
  - R1
  supported_by:
  - c_gpt4_icl_overall
  - c_gpt4_mt_overall
  - c_gpt35_icl_overall
  - c_gpt35_mt_overall
  - c_v7_icl_overall
  - c_v7_mt_overall
  - c_v13_icl_overall
  - c_v13_mt_overall
  page: 1
  contradicted_by:
  - r_gpt35_icl_email
  - r_v7_icl_summ
  - r_v7_mt_email
  - r_v13_icl_email
  - r_v13_icl_web
  - r_v13_icl_table
  - r_v13_icl_overall
  - r_v13_mt_email
- id: k2
  text: the white-box defence brings attack success to near zero
  location: abstract
  anchor: our white-box defense reduces the attack success rate to near-zero levels
  jobs:
  - R1
  supported_by:
  - c_v7_wb_bipia_overall
  - c_v7_wb_orig_overall
  - c_v7_wb_gpt4r_overall
  - c_v13_wb_bipia_overall
  - c_v13_wb_orig_overall
  - c_v13_wb_gpt4r_overall
  page: 1
  contradicted_by:
  - r_v7_wb_orig_code
- id: k3
  text: both defences preserve the output quality of the LLMs
  location: abstract
  anchor: all while preserving the output quality of LLMs
  jobs:
  - R2
  - general_utility
  supported_by:
  - c_gpt4_icl_rouge
  - c_gpt4_mt_rouge
  - c_gpt35_icl_rouge
  - c_gpt35_mt_rouge
  - c_v7_icl_rouge
  - c_v7_mt_rouge
  - c_v13_icl_rouge
  - c_v13_mt_rouge
  - c_v7_wb_bipia_rouge
  - c_v7_wb_bipia_mtbench
  - c_v7_wb_orig_rouge
  - c_v7_wb_orig_mtbench
  - c_v7_wb_gpt4r_rouge
  - c_v7_wb_gpt4r_mtbench
  - c_v13_wb_bipia_rouge
  - c_v13_wb_bipia_mtbench
  - c_v13_wb_orig_rouge
  - c_v13_wb_orig_mtbench
  - c_v13_wb_gpt4r_rouge
  - c_v13_wb_gpt4r_mtbench
  page: 1
  contradicted_by:
  - r_v7_wb_bipia_rouge
  - r_v7_wb_bipia_mtbench
  - r_v13_wb_bipia_mtbench
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 3 and 4 (per task and overall)
    R2clean: Tables 3 and 4 (ROUGE-1 recall on BIPIA-Clean)
    general_utility: Table 4 (MT-Bench, white-box only)

`cacheprune_arxiv2025` (2504.21228)

In [ ]:
%%writefile reports/records/cacheprune_arxiv2025.yaml
report:
  id: cacheprune_arxiv2025
  slug: '2504.21228'
  title: 'CachePrune: Teaching LLMs What Not to Follow via KV-Cache Editing'
  first_author: Rui Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2504.21228'
  version_read: arXiv v3
  first_date: '2025-04-29'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'Sections 3 to 4.2 and Appendix A: the Vanilla and CachePrune rows of Table 1 for all three models and all columns; the abstract claims (Delimiting, Datamarking, Sandwich and Encode_Base64 rows, Tables
    2 to 6, the adaptive attacks of Appendix F and Tables 13 to 14 not coded)'
defences:
- id: cp_l3
  name: CachePrune
  origin: this_report
  base_checkpoint: LLama3-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - neurons in the key and value cache of the context tokens pruned (0.5% of neurons)
  - mask chosen by a preferential attribution loss on 8 examples (poisoned against clean responses)
  layout: the original prompt (Vanilla) with the context span's position known; the pruning mask applied to the context's KV cache
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not applicable (no weights changed)
  anchor: By default, We experiment with N = 8 for neu- ral attribution
  signal_dependence_detail: the masking requires knowing the context boundary (p. 6); no run with a wrong or missing boundary
  pages:
    isolation: 13
- id: cp_m7
  name: CachePrune
  origin: this_report
  base_checkpoint: Mistral-7B-Instruct-V3.0
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - neurons in the key and value cache of the context tokens pruned (0.5% of neurons)
  - mask chosen by a preferential attribution loss on 8 examples (poisoned against clean responses)
  layout: the original prompt (Vanilla) with the context span's position known; the pruning mask applied to the context's KV cache
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not applicable (no weights changed)
  anchor: By default, We experiment with N = 8 for neu- ral attribution
  signal_dependence_detail: the masking requires knowing the context boundary (p. 6); no run with a wrong or missing boundary
  pages:
    isolation: 13
- id: cp_phi
  name: CachePrune
  origin: this_report
  base_checkpoint: Phi-3.5-mini-instruct (3.8B)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  components:
  - neurons in the key and value cache of the context tokens pruned (0.5% of neurons)
  - mask chosen by a preferential attribution loss on 8 examples (poisoned against clean responses)
  layout: the original prompt (Vanilla) with the context span's position known; the pruning mask applied to the context's KV cache
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not applicable (no weights changed)
  anchor: By default, We experiment with N = 8 for neu- ral attribution
  signal_dependence_detail: the masking requires knowing the context boundary (p. 6); no run with a wrong or missing boundary
  pages:
    isolation: 13
evaluations:
- id: ev_l3_none_squad_attack
  defence_id: none:LLama3-8B
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_l3_none_squad_clean
  defence_id: none:LLama3-8B
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_l3_none_hotpot_attack
  defence_id: none:LLama3-8B
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_l3_none_hotpot_clean
  defence_id: none:LLama3-8B
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_l3_none_wild_attack
  defence_id: none:LLama3-8B
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_l3_cp_squad_attack
  defence_id: cp_l3
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_l3_cp_squad_clean
  defence_id: cp_l3
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_l3_cp_hotpot_attack
  defence_id: cp_l3
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_l3_cp_hotpot_clean
  defence_id: cp_l3
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_l3_cp_wild_attack
  defence_id: cp_l3
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_m7_none_squad_attack
  defence_id: none:Mistral-7B-Instruct-V3.0
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_m7_none_squad_clean
  defence_id: none:Mistral-7B-Instruct-V3.0
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_m7_none_hotpot_attack
  defence_id: none:Mistral-7B-Instruct-V3.0
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_m7_none_hotpot_clean
  defence_id: none:Mistral-7B-Instruct-V3.0
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_m7_none_wild_attack
  defence_id: none:Mistral-7B-Instruct-V3.0
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_m7_cp_squad_attack
  defence_id: cp_m7
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_m7_cp_squad_clean
  defence_id: cp_m7
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_m7_cp_hotpot_attack
  defence_id: cp_m7
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_m7_cp_hotpot_clean
  defence_id: cp_m7
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_m7_cp_wild_attack
  defence_id: cp_m7
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_phi_none_squad_attack
  defence_id: none:Phi-3.5-mini-instruct (3.8B)
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_phi_none_squad_clean
  defence_id: none:Phi-3.5-mini-instruct (3.8B)
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_phi_none_hotpot_attack
  defence_id: none:Phi-3.5-mini-instruct (3.8B)
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_phi_none_hotpot_clean
  defence_id: none:Phi-3.5-mini-instruct (3.8B)
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_phi_none_wild_attack
  defence_id: none:Phi-3.5-mini-instruct (3.8B)
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  baseline_kind: off_the_shelf
- id: ev_phi_cp_squad_attack
  defence_id: cp_phi
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_phi_cp_squad_clean
  defence_id: cp_phi
  benchmark: SQuAD with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_phi_cp_hotpot_attack
  defence_id: cp_phi
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  injection_position: varied
  run: mean of 3 trials (CachePrune rows)
- id: ev_phi_cp_hotpot_clean
  defence_id: cp_phi
  benchmark: HotpotQA with instructions injected at the start, middle or end of the context (Abdelnabi et al. splits) (clean prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: F1 against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
- id: ev_phi_cp_wild_attack
  defence_id: cp_phi
  benchmark: WildChat dialogue summarization, a user question in the dialogue with an added focus sentence
  attacker_type: static
  attacker_access: black_box
  judge: proportion of poisoned responses (how a response is judged poisoned is not stated); LLM judge scores summary quality 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: cacheprune_arxiv2025
  page: 7
  decoding: greedy
  pages:
    benchmark: 6
    judge: 11
    decoding: 11
  run: mean of 3 trials (CachePrune rows)
results:
- id: r_l3_none_squad_asr
  evaluation_id: ev_l3_none_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '27.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_squad_f1c
  evaluation_id: ev_l3_none_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '28.20'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_squad_f1
  evaluation_id: ev_l3_none_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '19.56'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_hotpot_asr
  evaluation_id: ev_l3_none_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '69.01'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_hotpot_f1c
  evaluation_id: ev_l3_none_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '16.24'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_hotpot_f1
  evaluation_id: ev_l3_none_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '5.12'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_wild_asr
  evaluation_id: ev_l3_none_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '14.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_none_wild_gpt-score
  evaluation_id: ev_l3_none_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.32'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_l3_cp_squad_asr
  evaluation_id: ev_l3_cp_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '7.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.22 over 3 trials
- id: r_l3_cp_squad_f1c
  evaluation_id: ev_l3_cp_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '28.68'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.30 over 3 trials
- id: r_l3_cp_squad_f1
  evaluation_id: ev_l3_cp_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '22.84'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.49 over 3 trials
- id: r_l3_cp_hotpot_asr
  evaluation_id: ev_l3_cp_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15.23'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 1.56 over 3 trials
- id: r_l3_cp_hotpot_f1c
  evaluation_id: ev_l3_cp_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '16.21'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.61 over 3 trials
- id: r_l3_cp_hotpot_f1
  evaluation_id: ev_l3_cp_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '10.97'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.35 over 3 trials
- id: r_l3_cp_wild_asr
  evaluation_id: ev_l3_cp_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.41 over 3 trials
- id: r_l3_cp_wild_gpt-score
  evaluation_id: ev_l3_cp_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.32'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.10 over 3 trials
- id: r_m7_none_squad_asr
  evaluation_id: ev_m7_none_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '9.01'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_squad_f1c
  evaluation_id: ev_m7_none_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '22.78'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_squad_f1
  evaluation_id: ev_m7_none_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '19.04'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_hotpot_asr
  evaluation_id: ev_m7_none_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '25.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_hotpot_f1c
  evaluation_id: ev_m7_none_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '14.10'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_hotpot_f1
  evaluation_id: ev_m7_none_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '10.12'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_wild_asr
  evaluation_id: ev_m7_none_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_none_wild_gpt-score
  evaluation_id: ev_m7_none_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.88'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_m7_cp_squad_asr
  evaluation_id: ev_m7_cp_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.68'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.41 over 3 trials
- id: r_m7_cp_squad_f1c
  evaluation_id: ev_m7_cp_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '24.46'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.91 over 3 trials
- id: r_m7_cp_squad_f1
  evaluation_id: ev_m7_cp_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '23.10'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 1.32 over 3 trials
- id: r_m7_cp_hotpot_asr
  evaluation_id: ev_m7_cp_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.51'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 1.10 over 3 trials
- id: r_m7_cp_hotpot_f1c
  evaluation_id: ev_m7_cp_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '14.38'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.57 over 3 trials
- id: r_m7_cp_hotpot_f1
  evaluation_id: ev_m7_cp_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '13.32'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.42 over 3 trials
- id: r_m7_cp_wild_asr
  evaluation_id: ev_m7_cp_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.26 over 3 trials
- id: r_m7_cp_wild_gpt-score
  evaluation_id: ev_m7_cp_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.90'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.03 over 3 trials
- id: r_phi_none_squad_asr
  evaluation_id: ev_phi_none_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_squad_f1c
  evaluation_id: ev_phi_none_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '26.03'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_squad_f1
  evaluation_id: ev_phi_none_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '25.64'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_hotpot_asr
  evaluation_id: ev_phi_none_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '21.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_hotpot_f1c
  evaluation_id: ev_phi_none_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '14.14'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_hotpot_f1
  evaluation_id: ev_phi_none_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '7.69'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_wild_asr
  evaluation_id: ev_phi_none_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_none_wild_gpt-score
  evaluation_id: ev_phi_none_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.78'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_phi_cp_squad_asr
  evaluation_id: ev_phi_cp_squad_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.18 over 3 trials
- id: r_phi_cp_squad_f1c
  evaluation_id: ev_phi_cp_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '26.76'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.56 over 3 trials
- id: r_phi_cp_squad_f1
  evaluation_id: ev_phi_cp_squad_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '25.55'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.60 over 3 trials
- id: r_phi_cp_hotpot_asr
  evaluation_id: ev_phi_cp_hotpot_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.76'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.50 over 3 trials
- id: r_phi_cp_hotpot_f1c
  evaluation_id: ev_phi_cp_hotpot_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: F1(clean)
  value: '14.17'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.78 over 3 trials
- id: r_phi_cp_hotpot_f1
  evaluation_id: ev_phi_cp_hotpot_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: F1 (attack)
  value: '9.79'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 1.12 over 3 trials
- id: r_phi_cp_wild_asr
  evaluation_id: ev_phi_cp_wild_attack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.25 over 3 trials
- id: r_phi_cp_wild_gpt-score
  evaluation_id: ev_phi_cp_wild_attack
  quantity: R3cp
  coverage: proxy
  metric_as_printed: GPT-Score
  value: '3.60'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: ± 0.31 over 3 trials
comparisons:
- id: c_l3_hotpot_attack
  left: ev_l3_cp_hotpot_attack
  right: ev_l3_none_hotpot_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_hotpot_clean
  left: ev_l3_cp_hotpot_clean
  right: ev_l3_none_hotpot_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_squad_attack
  left: ev_l3_cp_squad_attack
  right: ev_l3_none_squad_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_squad_clean
  left: ev_l3_cp_squad_clean
  right: ev_l3_none_squad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_wild_attack
  left: ev_l3_cp_wild_attack
  right: ev_l3_none_wild_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_hotpot_attack
  left: ev_m7_cp_hotpot_attack
  right: ev_m7_none_hotpot_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_hotpot_clean
  left: ev_m7_cp_hotpot_clean
  right: ev_m7_none_hotpot_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_squad_attack
  left: ev_m7_cp_squad_attack
  right: ev_m7_none_squad_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_squad_clean
  left: ev_m7_cp_squad_clean
  right: ev_m7_none_squad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_wild_attack
  left: ev_m7_cp_wild_attack
  right: ev_m7_none_wild_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_hotpot_attack
  left: ev_phi_cp_hotpot_attack
  right: ev_phi_none_hotpot_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_hotpot_clean
  left: ev_phi_cp_hotpot_clean
  right: ev_phi_none_hotpot_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_squad_attack
  left: ev_phi_cp_squad_attack
  right: ev_phi_none_squad_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_squad_clean
  left: ev_phi_cp_squad_clean
  right: ev_phi_none_squad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_wild_attack
  left: ev_phi_cp_wild_attack
  right: ev_phi_none_wild_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: CachePrune significantly reduces attack success
  location: abstract
  anchor: CachePrune significantly reduces the attack success rate
  jobs:
  - R1
  supported_by:
  - c_l3_squad_attack
  - c_l3_hotpot_attack
  - c_l3_wild_attack
  - c_m7_squad_attack
  - c_m7_hotpot_attack
  - c_m7_wild_attack
  - c_phi_squad_attack
  - c_phi_hotpot_attack
  - c_phi_wild_attack
  page: 1
- id: k2
  text: while keeping the model's ability to follow user instructions
  location: abstract
  anchor: preserving the LLM’s ability to follow user instructions
  jobs:
  - R2
  supported_by:
  - c_l3_squad_clean
  - c_l3_hotpot_clean
  - c_m7_squad_clean
  - c_m7_hotpot_clean
  - c_phi_squad_clean
  - c_phi_hotpot_clean
  - c_l3_squad_attack
  - c_l3_hotpot_attack
  - c_m7_squad_attack
  - c_m7_hotpot_attack
  - c_phi_squad_attack
  - c_phi_hotpot_attack
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: proxy
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: Table 1 (F1 clean)
    R2attack: Table 1 (F1 attack)
    R3cp: Table 1 (GPT-Score of WildChat summaries)

`drip_arxiv2025` (2511.00447)

In [ ]:
%%writefile reports/records/drip_arxiv2025.yaml
report:
  id: drip_arxiv2025
  slug: '2511.00447'
  title: 'DRIP: Defending Prompt Injection via Token-wise Representation Editing and Residual Instruction Fusion'
  first_author: Ruofan Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2511.00447'
  version_read: arXiv v2
  first_date: '2025-11-01'
  read: F
  role: proposes
  code_release: demo site cited; code not checked
  later_version: DOI 10.1145/3830454.3832577 (from the search record; not read)
  extraction_scope: 'Sections 3 and 4.1 to 4.3: the Undefended, SecAlign and Ours rows of Tables 3, 4 and 6 and the Naive, Hackaprompt, GCG and NeuralExec rows of Table 5; the abstract claims (StruQ, ISE and PFT rows, family
    averages of Table 5, Figure 8, Table 7 ablation and Appendix tables not coded)'
defences:
- id: drip_l8
  name: DRIP
  origin: this_report
  base_checkpoint: LLaMA-8B (as printed; Llama 3 chat template)
  axis1:
  - chat_roles
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: partly
  trains_carry_through: yes_by_preference
  page: 6
  components:
  - 'de-instruction shift: a learned token-wise linear edit of the data tokens'' embeddings'
  - 'instruction fusion: the last instruction token''s hidden state added to the output state before the LM head'
  - 'DPO with LoRA on curated SEP-train pairs: injected tasks resampled from SQuAD so the same instruction strings also appear as top-level instructions with other data'
  - preferred responses process the whole data section, the injected sentence included (Figure 4 translates it; the prompt adds a do-not-omit instruction)
  layout: instruction in the system turn, data in the user turn, response in the assistant turn
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: not stated
  anchor: 'Experiments use two widely adopted decoder-only backbones: LLaMA-8B [19] and Mistral-7B'
  pages:
    components: 4
    trains_instruction_pair: 5
    trains_carry_through: 6
    layout: 5
    isolation: 12
- id: secalign_l8
  name: SecAlign (baseline run by the DRIP authors)
  origin: other_report
  base_checkpoint: LLaMA-8B (as printed; Llama 3 chat template)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Extends StruQ by replacing the SFT loss with a preference-based DPO objective
- id: drip_m7
  name: DRIP
  origin: this_report
  base_checkpoint: Mistral-7B (as printed)
  axis1:
  - chat_roles
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: partly
  trains_carry_through: yes_by_preference
  page: 6
  components:
  - 'de-instruction shift: a learned token-wise linear edit of the data tokens'' embeddings'
  - 'instruction fusion: the last instruction token''s hidden state added to the output state before the LM head'
  - 'DPO with LoRA on curated SEP-train pairs: injected tasks resampled from SQuAD so the same instruction strings also appear as top-level instructions with other data'
  - preferred responses process the whole data section, the injected sentence included (Figure 4 translates it; the prompt adds a do-not-omit instruction)
  layout: instruction in the system turn, data in the user turn, response in the assistant turn
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: not stated
  anchor: 'Experiments use two widely adopted decoder-only backbones: LLaMA-8B [19] and Mistral-7B'
  pages:
    components: 4
    trains_instruction_pair: 5
    trains_carry_through: 6
    layout: 5
    isolation: 12
- id: secalign_m7
  name: SecAlign (baseline run by the DRIP authors)
  origin: other_report
  base_checkpoint: Mistral-7B (as printed)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Extends StruQ by replacing the SFT loss with a preference-based DPO objective
evaluations:
- id: ev_l8_none_sep
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l8_sec_sep
  defence_id: secalign_l8
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
- id: ev_l8_drip_sep
  defence_id: drip_l8
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
- id: ev_m7_none_sep
  defence_id: none:Mistral-7B (as printed)
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m7_sec_sep
  defence_id: secalign_m7
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
- id: ev_m7_drip_sep
  defence_id: drip_m7
  benchmark: SEP benchmark (9,160 tuples, injected task in the data and as the instruction)
  attacker_type: static
  attacker_access: black_box
  judge: witness string emitted in the instruction placement and suppressed in the data placement
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  sample_size: 9,160
  decoding: not reported
- id: ev_l8_none_injecagent
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l8_sec_injecagent
  defence_id: secalign_l8
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
- id: ev_l8_drip_injecagent
  defence_id: drip_l8
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
- id: ev_m7_none_injecagent
  defence_id: none:Mistral-7B (as printed)
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m7_sec_injecagent
  defence_id: secalign_m7
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
- id: ev_m7_drip_injecagent
  defence_id: drip_m7
  benchmark: InjecAgent, Enhanced setting (1,054 cases)
  attacker_type: static
  attacker_access: black_box
  judge: agent executes the malicious API
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 7
  attacker_budget: Enhanced setting adversarial prefix
  sample_size: 1,054
  decoding: not reported
- id: ev_l8_none_naive
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_l8_sec_naive
  defence_id: secalign_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_drip_naive
  defence_id: drip_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_none_naive
  defence_id: none:Mistral-7B (as printed)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_m7_sec_naive
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_drip_naive
  defence_id: drip_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: static
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Naive
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_none_hackaprompt
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_l8_sec_hackaprompt
  defence_id: secalign_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_drip_hackaprompt
  defence_id: drip_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_none_hackaprompt
  defence_id: none:Mistral-7B (as printed)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_m7_sec_hackaprompt
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_drip_hackaprompt
  defence_id: drip_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: human
  attacker_access: black_box
  judge: reply equals hacked after trimming whitespace
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: Hackaprompt
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_none_gcg
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_l8_sec_gcg
  defence_id: secalign_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_drip_gcg
  defence_id: drip_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_none_gcg
  defence_id: none:Mistral-7B (as printed)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_m7_sec_gcg
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_drip_gcg
  defence_id: drip_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: GCG sample-specific prefix and suffix
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_none_neuralexec
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_l8_sec_neuralexec
  defence_id: secalign_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_drip_neuralexec
  defence_id: drip_l8
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_none_neuralexec
  defence_id: none:Mistral-7B (as printed)
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
  baseline_kind: off_the_shelf
- id: ev_m7_sec_neuralexec
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_m7_drip_neuralexec
  defence_id: drip_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'hacked'
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply starts with hacked, case-insensitive
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 8
  attacker_budget: NeuralExec model-universal trigger
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
- id: ev_l8_none_ifeval
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l8_sec_ifeval
  defence_id: secalign_l8
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
- id: ev_l8_drip_ifeval
  defence_id: drip_l8
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
- id: ev_m7_none_ifeval
  defence_id: none:Mistral-7B (as printed)
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m7_sec_ifeval
  defence_id: secalign_m7
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
- id: ev_m7_drip_ifeval
  defence_id: drip_m7
  benchmark: IFEval (541 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: official rule-based checks, strict instruction-level accuracy
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '541'
  decoding: not reported
- id: ev_l8_none_alpacaeval
  defence_id: none:LLaMA-8B (as printed; Llama 3 chat template)
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l8_sec_alpacaeval
  defence_id: secalign_l8
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
- id: ev_l8_drip_alpacaeval
  defence_id: drip_l8
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
- id: ev_m7_none_alpacaeval
  defence_id: none:Mistral-7B (as printed)
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m7_sec_alpacaeval
  defence_id: secalign_m7
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
- id: ev_m7_drip_alpacaeval
  defence_id: drip_m7
  benchmark: AlpacaEval 2.0 (805 prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge win rate against GPT-4 reference outputs, ties as half
  layout_same_as_defence: 'yes'
  measured_by: drip_arxiv2025
  page: 10
  sample_size: '805'
  decoding: not reported
results:
- id: r_l8_none_sep
  evaluation_id: ev_l8_none_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '21.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l8_sec_sep
  evaluation_id: ev_l8_sec_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '31.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l8_drip_sep
  evaluation_id: ev_l8_drip_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '80.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m7_none_sep
  evaluation_id: ev_m7_none_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '20.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m7_sec_sep
  evaluation_id: ev_m7_sec_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '58.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m7_drip_sep
  evaluation_id: ev_m7_drip_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP %
  value: '70.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l8_none_injecagent
  evaluation_id: ev_l8_none_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '64.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_l8_sec_injecagent
  evaluation_id: ev_l8_sec_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_l8_drip_injecagent
  evaluation_id: ev_l8_drip_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_m7_none_injecagent
  evaluation_id: ev_m7_none_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '30.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_m7_sec_injecagent
  evaluation_id: ev_m7_sec_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_m7_drip_injecagent
  evaluation_id: ev_m7_drip_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR %
  value: '1.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_l8_none_naive
  evaluation_id: ev_l8_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '5.74'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_sec_naive
  evaluation_id: ev_l8_sec_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_drip_naive
  evaluation_id: ev_l8_drip_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_none_naive
  evaluation_id: ev_m7_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '2.39'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_sec_naive
  evaluation_id: ev_m7_sec_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_drip_naive
  evaluation_id: ev_m7_drip_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_none_hackaprompt
  evaluation_id: ev_l8_none_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '23.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_sec_hackaprompt
  evaluation_id: ev_l8_sec_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_drip_hackaprompt
  evaluation_id: ev_l8_drip_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_none_hackaprompt
  evaluation_id: ev_m7_none_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '38.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_sec_hackaprompt
  evaluation_id: ev_m7_sec_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_drip_hackaprompt
  evaluation_id: ev_m7_drip_hackaprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Hackaprompt
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_none_gcg
  evaluation_id: ev_l8_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '98.08'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_sec_gcg
  evaluation_id: ev_l8_sec_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '66.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_drip_gcg
  evaluation_id: ev_l8_drip_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '1.06'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_none_gcg
  evaluation_id: ev_m7_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_sec_gcg
  evaluation_id: ev_m7_sec_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '98.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_drip_gcg
  evaluation_id: ev_m7_drip_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '3.37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_none_neuralexec
  evaluation_id: ev_l8_none_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '12.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_sec_neuralexec
  evaluation_id: ev_l8_sec_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_drip_neuralexec
  evaluation_id: ev_l8_drip_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_none_neuralexec
  evaluation_id: ev_m7_none_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '51.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_sec_neuralexec
  evaluation_id: ev_m7_sec_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '2.88'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_m7_drip_neuralexec
  evaluation_id: ev_m7_drip_neuralexec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NeuralExec
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_l8_none_ifeval
  evaluation_id: ev_l8_none_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '72.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_l8_sec_ifeval
  evaluation_id: ev_l8_sec_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '65.47'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_l8_drip_ifeval
  evaluation_id: ev_l8_drip_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '76.02'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_none_ifeval
  evaluation_id: ev_m7_none_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '58.51'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_sec_ifeval
  evaluation_id: ev_m7_sec_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '48.68'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_drip_ifeval
  evaluation_id: ev_m7_drip_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval (%)
  value: '60.07'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_l8_none_alpacaeval
  evaluation_id: ev_l8_none_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '85.37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_l8_sec_alpacaeval
  evaluation_id: ev_l8_sec_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '64.64'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_l8_drip_alpacaeval
  evaluation_id: ev_l8_drip_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '83.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_none_alpacaeval
  evaluation_id: ev_m7_none_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '86.39'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_sec_alpacaeval
  evaluation_id: ev_m7_sec_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '72.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
- id: r_m7_drip_alpacaeval
  evaluation_id: ev_m7_drip_alpacaeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval-2.0 (%)
  value: '82.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 6
comparisons:
- id: c_l8_sep_drip_none
  left: ev_l8_drip_sep
  right: ev_l8_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l8_sep_drip_sec
  left: ev_l8_drip_sep
  right: ev_l8_sec_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 7
- id: c_m7_sep_drip_none
  left: ev_m7_drip_sep
  right: ev_m7_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_sep_drip_sec
  left: ev_m7_drip_sep
  right: ev_m7_sec_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 7
- id: c_l8_injecagent_drip_none
  left: ev_l8_drip_injecagent
  right: ev_l8_none_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l8_injecagent_drip_sec
  left: ev_l8_drip_injecagent
  right: ev_l8_sec_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 7
- id: c_m7_injecagent_drip_none
  left: ev_m7_drip_injecagent
  right: ev_m7_none_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m7_injecagent_drip_sec
  left: ev_m7_drip_injecagent
  right: ev_m7_sec_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 7
- id: c_l8_naive_drip_none
  left: ev_l8_drip_naive
  right: ev_l8_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l8_naive_drip_sec
  left: ev_l8_drip_naive
  right: ev_l8_sec_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_m7_naive_drip_none
  left: ev_m7_drip_naive
  right: ev_m7_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_m7_naive_drip_sec
  left: ev_m7_drip_naive
  right: ev_m7_sec_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_l8_hackaprompt_drip_none
  left: ev_l8_drip_hackaprompt
  right: ev_l8_none_hackaprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l8_hackaprompt_drip_sec
  left: ev_l8_drip_hackaprompt
  right: ev_l8_sec_hackaprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_m7_hackaprompt_drip_none
  left: ev_m7_drip_hackaprompt
  right: ev_m7_none_hackaprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_m7_hackaprompt_drip_sec
  left: ev_m7_drip_hackaprompt
  right: ev_m7_sec_hackaprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_l8_gcg_drip_none
  left: ev_l8_drip_gcg
  right: ev_l8_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l8_gcg_drip_sec
  left: ev_l8_drip_gcg
  right: ev_l8_sec_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_m7_gcg_drip_none
  left: ev_m7_drip_gcg
  right: ev_m7_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_m7_gcg_drip_sec
  left: ev_m7_drip_gcg
  right: ev_m7_sec_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_l8_neuralexec_drip_none
  left: ev_l8_drip_neuralexec
  right: ev_l8_none_neuralexec
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l8_neuralexec_drip_sec
  left: ev_l8_drip_neuralexec
  right: ev_l8_sec_neuralexec
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_m7_neuralexec_drip_none
  left: ev_m7_drip_neuralexec
  right: ev_m7_none_neuralexec
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_m7_neuralexec_drip_sec
  left: ev_m7_drip_neuralexec
  right: ev_m7_sec_neuralexec
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 8
- id: c_l8_ifeval_drip_none
  left: ev_l8_drip_ifeval
  right: ev_l8_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_l8_ifeval_drip_sec
  left: ev_l8_drip_ifeval
  right: ev_l8_sec_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 10
- id: c_m7_ifeval_drip_none
  left: ev_m7_drip_ifeval
  right: ev_m7_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m7_ifeval_drip_sec
  left: ev_m7_drip_ifeval
  right: ev_m7_sec_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 10
- id: c_l8_alpacaeval_drip_none
  left: ev_l8_drip_alpacaeval
  right: ev_l8_none_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_l8_alpacaeval_drip_sec
  left: ev_l8_drip_alpacaeval
  right: ev_l8_sec_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 10
- id: c_m7_alpacaeval_drip_none
  left: ev_m7_drip_alpacaeval
  right: ev_m7_none_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m7_alpacaeval_drip_sec
  left: ev_m7_drip_alpacaeval
  right: ev_m7_sec_alpacaeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: DRIP against SecAlign trained by the same authors on the same base'
  page: 10
claims:
- id: k1
  text: role separation score improves by 12 to 49 points over prior defences
  location: abstract
  anchor: improves role separation score by 12–49%
  jobs:
  - R1
  - R2
  supported_by:
  - c_l8_sep_drip_sec
  - c_m7_sep_drip_sec
  page: 1
- id: k2
  text: attack success under adaptive attacks falls by over 66%
  location: abstract
  anchor: duces attack success rate by over 66% for adaptive attacks
  jobs:
  - R1
  supported_by:
  - c_l8_gcg_drip_sec
  - c_m7_gcg_drip_sec
  - c_l8_gcg_drip_none
  - c_m7_gcg_drip_none
  page: 1
- id: k3
  text: utility on par with the undefended model
  location: abstract
  anchor: achieves utility on par with the undefended model
  jobs:
  - general_utility
  supported_by:
  - c_l8_ifeval_drip_none
  - c_m7_ifeval_drip_none
  - c_l8_alpacaeval_drip_none
  - c_m7_alpacaeval_drip_none
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 4 and 5
    other: Table 3 (SEP score)
    general_utility: Table 6 (IFEval, AlpacaEval 2.0)

`fath_arxiv2024` (2410.21492)

In [ ]:
%%writefile reports/records/fath_arxiv2024.yaml
report:
  id: fath_arxiv2024
  slug: '2410.21492'
  title: 'FATH: Authentication-based Test-time Defense against Indirect Prompt Injection Attacks'
  first_author: Jiongxiao Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2410.21492'
  version_read: arXiv v2
  first_date: '2024-10-28'
  read: F
  role: proposes
  code_release: https://github.com/Jayfeather1024/FATH
  extraction_scope: 'Sections 4 and 5.1 to 5.6: the No Defense and FATH rows of Table 2 (Judge Score, Combined Attack and Adaptive Attack columns), Table 3, the optimisation-attack numbers in Section 5.5 and the w/o Authentication
    Tags row of Table 4; the abstract claim (other baselines, Naive to Fake Completion columns and the appendix not coded)'
defences:
- id: fath_l3
  name: FATH
  origin: this_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - instruction and external text wrapped in fresh hash-based tags
  - 'security policy: answer every instruction, each under its own tag'
  - rule-based parsing returns only the text under the authorised-response tag
  - chain-of-thought and in-context examples in the prompt
  layout: 'one user message: instruction and external text each between hash-based tags'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Finally, an authentication verification process
  signal_dependence_detail: 'Table 4 (GPT3.5) removes the authentication tags: adaptive ASR 0.06 to 0.18 against 0.00'
  pages:
    signal_dependence: 9
    isolation: 9
- id: fath_g35
  name: FATH
  origin: this_report
  base_checkpoint: gpt-3.5-turbo
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - instruction and external text wrapped in fresh hash-based tags
  - 'security policy: answer every instruction, each under its own tag'
  - rule-based parsing returns only the text under the authorised-response tag
  - chain-of-thought and in-context examples in the prompt
  layout: 'one user message: instruction and external text each between hash-based tags'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Finally, an authentication verification process
  signal_dependence_detail: 'Table 4 (GPT3.5) removes the authentication tags: adaptive ASR 0.06 to 0.18 against 0.00'
  pages:
    signal_dependence: 9
    isolation: 9
- id: fath_notags_g35
  name: FATH without authentication tags (ablation)
  origin: this_report
  base_checkpoint: gpt-3.5-turbo
  axis1:
  - no_signal
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  anchor: w/o Authentication Tags
evaluations:
- id: ev_l3_none_judge
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ target instructions without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-3.5 rates each answer 1 to 10 (LLM-as-a-Judge), averaged
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_comb_url
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_comb_qa
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_comb_clf
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_adapt_url
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_adapt_qa
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_none_adapt_clf
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_fath_judge
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ target instructions without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-3.5 rates each answer 1 to 10 (LLM-as-a-Judge), averaged
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_comb_url
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_comb_qa
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_comb_clf
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_adapt_url
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_adapt_qa
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_fath_adapt_clf
  defence_id: fath_l3
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_none_judge
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ target instructions without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-3.5 rates each answer 1 to 10 (LLM-as-a-Judge), averaged
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_comb_url
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_comb_qa
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_comb_clf
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_adapt_url
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_adapt_qa
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_none_adapt_clf
  defence_id: none:gpt-3.5-turbo
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_fath_judge
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ target instructions without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-3.5 rates each answer 1 to 10 (LLM-as-a-Judge), averaged
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_comb_url
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_comb_qa
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_comb_clf
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack (escape characters, context ignoring, fake completion)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_adapt_url
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_adapt_qa
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_fath_adapt_clf
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack built on the Combined Attack with knowledge of the defence prompt
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_none_ia_comb
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_fath_ia_comb
  defence_id: fath_l3
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_none_ia_adapt
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_l3_fath_ia_adapt
  defence_id: fath_l3
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack against FATH
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_none_ia_comb
  defence_id: none:gpt-3.5-turbo
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_fath_ia_comb
  defence_id: fath_g35
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: static
  attacker_access: black_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_g35_none_ia_adapt
  defence_id: none:gpt-3.5-turbo
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: Combined Attack
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g35_fath_ia_adapt
  defence_id: fath_g35
  benchmark: InjecAgent, direct-harm attack intention (510 examples)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 8
  attacker_budget: hand-written adaptive attack against FATH
  sample_size: '510'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_l3_none_opt
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: OpenPromptInjection+, injection Print exactly "Hacked"! with an optimised prefix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply is the target output Hacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: gradient-optimised prefix (Liu et al. 2024 framework); against FATH, expectation over five tag sets
  decoding: model default generation parameters
  pages:
    benchmark: 9
    judge: 9
  baseline_kind: off_the_shelf
- id: ev_l3_fath_opt
  defence_id: fath_l3
  benchmark: OpenPromptInjection+, injection Print exactly "Hacked"! with an optimised prefix
  attacker_type: optimisation
  attacker_access: white_box
  judge: reply is the target output Hacked
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: gradient-optimised prefix (Liu et al. 2024 framework); against FATH, expectation over five tag sets
  decoding: model default generation parameters
  pages:
    benchmark: 9
    judge: 9
- id: ev_t4_notags_url
  defence_id: fath_notags_g35
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against this setting (Appendix E.2)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_t4_fath_url
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (URL injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against FATH
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_t4_notags_qa
  defence_id: fath_notags_g35
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against this setting (Appendix E.2)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_t4_fath_qa
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (QA injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against FATH
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_t4_notags_clf
  defence_id: fath_notags_g35
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against this setting (Appendix E.2)
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
- id: ev_t4_fath_clf
  defence_id: fath_g35
  benchmark: OpenPromptInjection+ (CLF injection task), 100 Stanford Alpaca target instructions
  attacker_type: adaptive
  attacker_access: grey_box
  judge: proportion of examples successfully attacked (criterion per injection task not stated in the pages read)
  layout_same_as_defence: 'yes'
  measured_by: fath_arxiv2024
  page: 9
  attacker_budget: hand-written adaptive attack against FATH
  sample_size: '100'
  decoding: model default generation parameters
  pages:
    benchmark: 7
    judge: 7
    decoding: 7
results:
- id: r_l3_none_judge
  evaluation_id: ev_l3_none_judge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Judge Score
  value: '8.31'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_comb_url
  evaluation_id: ev_l3_none_comb_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, URL
  value: '0.60'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_comb_qa
  evaluation_id: ev_l3_none_comb_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, QA
  value: '0.98'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_comb_clf
  evaluation_id: ev_l3_none_comb_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, CLF
  value: '0.72'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_adapt_url
  evaluation_id: ev_l3_none_adapt_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.60'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_adapt_qa
  evaluation_id: ev_l3_none_adapt_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.98'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_adapt_clf
  evaluation_id: ev_l3_none_adapt_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.72'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_judge
  evaluation_id: ev_l3_fath_judge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Judge Score
  value: '6.73'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_comb_url
  evaluation_id: ev_l3_fath_comb_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, URL
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_comb_qa
  evaluation_id: ev_l3_fath_comb_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, QA
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_comb_clf
  evaluation_id: ev_l3_fath_comb_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, CLF
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_adapt_url
  evaluation_id: ev_l3_fath_adapt_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.26'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_adapt_qa
  evaluation_id: ev_l3_fath_adapt_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.34'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_fath_adapt_clf
  evaluation_id: ev_l3_fath_adapt_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.31'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_judge
  evaluation_id: ev_g35_none_judge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Judge Score
  value: '7.94'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_comb_url
  evaluation_id: ev_g35_none_comb_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, URL
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_comb_qa
  evaluation_id: ev_g35_none_comb_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, QA
  value: '0.70'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_comb_clf
  evaluation_id: ev_g35_none_comb_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, CLF
  value: '0.84'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_adapt_url
  evaluation_id: ev_g35_none_adapt_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_adapt_qa
  evaluation_id: ev_g35_none_adapt_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.70'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_none_adapt_clf
  evaluation_id: ev_g35_none_adapt_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.84'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_judge
  evaluation_id: ev_g35_fath_judge
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Judge Score
  value: '6.91'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_comb_url
  evaluation_id: ev_g35_fath_comb_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, URL
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_comb_qa
  evaluation_id: ev_g35_fath_comb_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, QA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_comb_clf
  evaluation_id: ev_g35_fath_comb_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack, CLF
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_adapt_url
  evaluation_id: ev_g35_fath_adapt_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_adapt_qa
  evaluation_id: ev_g35_fath_adapt_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_g35_fath_adapt_clf
  evaluation_id: ev_g35_fath_adapt_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l3_none_ia_comb
  evaluation_id: ev_l3_none_ia_comb
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack
  value: '99.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_fath_ia_comb
  evaluation_id: ev_l3_fath_ia_comb
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_ia_adapt
  evaluation_id: ev_l3_none_ia_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack
  value: '99.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_fath_ia_adapt
  evaluation_id: ev_l3_fath_ia_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_g35_none_ia_comb
  evaluation_id: ev_g35_none_ia_comb
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_g35_fath_ia_comb
  evaluation_id: ev_g35_fath_ia_comb
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Combined Attack
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_g35_none_ia_adapt
  evaluation_id: ev_g35_none_ia_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_g35_fath_ia_adapt
  evaluation_id: ev_g35_fath_ia_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_opt
  evaluation_id: ev_l3_none_opt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 9
  location: Section 5.5
- id: r_l3_fath_opt
  evaluation_id: ev_l3_fath_opt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 9
  location: Section 5.5
- id: r_t4_notags_url
  evaluation_id: ev_t4_notags_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_t4_fath_url
  evaluation_id: ev_t4_fath_url
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, URL
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_t4_notags_qa
  evaluation_id: ev_t4_notags_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_t4_fath_qa
  evaluation_id: ev_t4_fath_qa
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, QA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_t4_notags_clf
  evaluation_id: ev_t4_notags_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.18'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_t4_fath_clf
  evaluation_id: ev_t4_fath_clf
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate, Adaptive Attack, CLF
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
comparisons:
- id: c_l3_judge
  left: ev_l3_fath_judge
  right: ev_l3_none_judge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_comb_url
  left: ev_l3_fath_comb_url
  right: ev_l3_none_comb_url
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_comb_qa
  left: ev_l3_fath_comb_qa
  right: ev_l3_none_comb_qa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_comb_clf
  left: ev_l3_fath_comb_clf
  right: ev_l3_none_comb_clf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_adapt_url
  left: ev_l3_fath_adapt_url
  right: ev_l3_none_adapt_url
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_adapt_qa
  left: ev_l3_fath_adapt_qa
  right: ev_l3_none_adapt_qa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_adapt_clf
  left: ev_l3_fath_adapt_clf
  right: ev_l3_none_adapt_clf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_judge
  left: ev_g35_fath_judge
  right: ev_g35_none_judge
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_comb_url
  left: ev_g35_fath_comb_url
  right: ev_g35_none_comb_url
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_comb_qa
  left: ev_g35_fath_comb_qa
  right: ev_g35_none_comb_qa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_comb_clf
  left: ev_g35_fath_comb_clf
  right: ev_g35_none_comb_clf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_adapt_url
  left: ev_g35_fath_adapt_url
  right: ev_g35_none_adapt_url
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_adapt_qa
  left: ev_g35_fath_adapt_qa
  right: ev_g35_none_adapt_qa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_adapt_clf
  left: ev_g35_fath_adapt_clf
  right: ev_g35_none_adapt_clf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_ia_comb
  left: ev_l3_fath_ia_comb
  right: ev_l3_none_ia_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_ia_adapt
  left: ev_l3_fath_ia_adapt
  right: ev_l3_none_ia_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_ia_comb
  left: ev_g35_fath_ia_comb
  right: ev_g35_none_ia_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g35_ia_adapt
  left: ev_g35_fath_ia_adapt
  right: ev_g35_none_ia_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_opt
  left: ev_l3_fath_opt
  right: ev_l3_none_opt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_t4_url
  left: ev_t4_fath_url
  right: ev_t4_notags_url
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hash-based authentication tags (FATH against FATH with the tags removed), each under its own adaptive attack
  page: 9
- id: c_t4_qa
  left: ev_t4_fath_qa
  right: ev_t4_notags_qa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hash-based authentication tags (FATH against FATH with the tags removed), each under its own adaptive attack
  page: 9
- id: c_t4_clf
  left: ev_t4_fath_clf
  right: ev_t4_notags_clf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hash-based authentication tags (FATH against FATH with the tags removed), each under its own adaptive attack
  page: 9
claims:
- id: k1
  text: state-of-the-art defence against indirect prompt injection on Llama3 and GPT3.5 across attack methods
  location: abstract
  anchor: achieving state-of-the-art performance under Llama3 and GPT3.5 models across various attack methods
  jobs:
  - R1
  supported_by:
  - c_l3_comb_url
  - c_l3_comb_qa
  - c_l3_comb_clf
  - c_l3_adapt_url
  - c_l3_adapt_qa
  - c_l3_adapt_clf
  - c_g35_comb_url
  - c_g35_comb_qa
  - c_g35_comb_clf
  - c_g35_adapt_url
  - c_g35_adapt_qa
  - c_g35_adapt_clf
  - c_l3_ia_comb
  - c_l3_ia_adapt
  - c_g35_ia_comb
  - c_g35_ia_adapt
  - c_l3_opt
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 2, 3 and 4, Section 5.5
    R2clean: Table 2 (Judge Score, GPT-3.5 rating of answers without attacks)
schema_gaps:
- where: results r_l3_none_ia_*, r_g35_none_ia_*
  note: Table 3 prints the Llama3 no-defence ASR as a percentage (99.3) and the GPT3.5 one as a fraction (1.00); each is coded in its printed unit

`rvr_arxiv2025` (2504.20472)

In [ ]:
%%writefile reports/records/rvr_arxiv2025.yaml
report:
  id: rvr_arxiv2025
  slug: '2504.20472'
  title: 'Robustness via Referencing: Defending against Prompt Injection Attacks by Referencing the Executed Instruction'
  first_author: Yulin Chen
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2504.20472'
  version_read: arXiv v2
  first_date: '2025-04-29'
  read: F
  role: proposes
  code_release: stated as public in a footnote; not checked
  later_version: Findings of ACL 2026, DOI 10.18653/v1/2026.findings-acl.61 (from the search record; not read)
  extraction_scope: 'Sections 4 and 5.1 to 5.3 and Appendix B: the None, StruQ and Ours rows of Table 2 (Inj-SQuAD, indirect) and the None and Ours rows of Table 3; the abstract claims (Table 1 direct scenario, other prompting
    baselines, Tables 4 to 6 and 9 to 12 not coded)'
defences:
- id: rvr_l3
  name: Robustness via Referencing (instruction referencing with filtering)
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - data split into lines of at most K words, each prefixed with a tag [L X]; the instruction placed as line 1
  - instruction and data enclosed in <Instruction Area> and <Data Area> markers
  - prompt asks the model to name the tag and instruction it executes for each answer; two in-context examples
  - filter keeps only the answer referencing [L 1]
  layout: 'one prompt: instruction area, then the tagged data lines in a data area'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The entire defense pipeline consists of three se- quential steps
  pages:
    components: 5
    isolation: 17
- id: struq_l3
  name: StruQ (baseline run by the authors)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: To compare with fine-tuning method, we select StruQ
- id: rvr_q2
  name: Robustness via Referencing (instruction referencing with filtering)
  origin: this_report
  base_checkpoint: Qwen2-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - data split into lines of at most K words, each prefixed with a tag [L X]; the instruction placed as line 1
  - instruction and data enclosed in <Instruction Area> and <Data Area> markers
  - prompt asks the model to name the tag and instruction it executes for each answer; two in-context examples
  - filter keeps only the answer referencing [L 1]
  layout: 'one prompt: instruction area, then the tagged data lines in a data area'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The entire defense pipeline consists of three se- quential steps
  pages:
    components: 5
    isolation: 17
- id: struq_q2
  name: StruQ (baseline run by the authors)
  origin: other_report
  base_checkpoint: Qwen2-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: To compare with fine-tuning method, we select StruQ
- id: rvr_l31
  name: Robustness via Referencing (instruction referencing with filtering)
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - data split into lines of at most K words, each prefixed with a tag [L X]; the instruction placed as line 1
  - instruction and data enclosed in <Instruction Area> and <Data Area> markers
  - prompt asks the model to name the tag and instruction it executes for each answer; two in-context examples
  - filter keeps only the answer referencing [L 1]
  layout: 'one prompt: instruction area, then the tagged data lines in a data area'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The entire defense pipeline consists of three se- quential steps
  pages:
    components: 5
    isolation: 17
- id: struq_l31
  name: StruQ (baseline run by the authors)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: To compare with fine-tuning method, we select StruQ
evaluations:
- id: ev_l3_none_naive
  defence_id: none:Llama3-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_ignore
  defence_id: none:Llama3-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_escape
  defence_id: none:Llama3-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_fakecom
  defence_id: none:Llama3-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_combined
  defence_id: none:Llama3-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_struq_naive
  defence_id: struq_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_struq_ignore
  defence_id: struq_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_struq_escape
  defence_id: struq_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_struq_fakecom
  defence_id: struq_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_struq_combined
  defence_id: struq_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_naive
  defence_id: rvr_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_ignore
  defence_id: rvr_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_escape
  defence_id: rvr_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_fakecom
  defence_id: rvr_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_combined
  defence_id: rvr_l3
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_none_util_squad
  defence_id: none:Llama3-8B-Instruct
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_util_triviaqa
  defence_id: none:Llama3-8B-Instruct
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_none_util_sst2
  defence_id: none:Llama3-8B-Instruct
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l3_rvr_util_squad
  defence_id: rvr_l3
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_util_triviaqa
  defence_id: rvr_l3
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l3_rvr_util_sst2
  defence_id: rvr_l3
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_none_naive
  defence_id: none:Qwen2-7B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_ignore
  defence_id: none:Qwen2-7B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_escape
  defence_id: none:Qwen2-7B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_fakecom
  defence_id: none:Qwen2-7B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_combined
  defence_id: none:Qwen2-7B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_struq_naive
  defence_id: struq_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_struq_ignore
  defence_id: struq_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_struq_escape
  defence_id: struq_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_struq_fakecom
  defence_id: struq_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_struq_combined
  defence_id: struq_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_naive
  defence_id: rvr_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_ignore
  defence_id: rvr_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_escape
  defence_id: rvr_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_fakecom
  defence_id: rvr_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_combined
  defence_id: rvr_q2
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_none_util_squad
  defence_id: none:Qwen2-7B-Instruct
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_util_triviaqa
  defence_id: none:Qwen2-7B-Instruct
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_none_util_sst2
  defence_id: none:Qwen2-7B-Instruct
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_q2_rvr_util_squad
  defence_id: rvr_q2
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_util_triviaqa
  defence_id: rvr_q2
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_q2_rvr_util_sst2
  defence_id: rvr_q2
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_none_naive
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_ignore
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_escape
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_fakecom
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_combined
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_struq_naive
  defence_id: struq_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_struq_ignore
  defence_id: struq_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_struq_escape
  defence_id: struq_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_struq_fakecom
  defence_id: struq_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_struq_combined
  defence_id: struq_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_naive
  defence_id: rvr_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Naive attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_ignore
  defence_id: rvr_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Ignore attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_escape
  defence_id: rvr_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Escape attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_fakecom
  defence_id: rvr_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_combined
  defence_id: rvr_l31
  benchmark: Inj-SQuAD (900 samples, injections for phishing, advertisement and propaganda)
  attacker_type: static
  attacker_access: black_box
  judge: response contains the answer to the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 6
  attacker_budget: Combined attack
  sample_size: '900'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_none_util_squad
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_util_triviaqa
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_util_sst2
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_l31_rvr_util_squad
  defence_id: rvr_l31
  benchmark: SQuAD without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_util_triviaqa
  defence_id: rvr_l31
  benchmark: TriviaQA without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
- id: ev_l31_rvr_util_sst2
  defence_id: rvr_l31
  benchmark: SST2 without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: rvr_arxiv2025
  page: 8
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 13
results:
- id: r_l3_none_naive
  evaluation_id: ev_l3_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '53.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_ignore
  evaluation_id: ev_l3_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '73.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_escape
  evaluation_id: ev_l3_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '75.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_fakecom
  evaluation_id: ev_l3_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '84.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_combined
  evaluation_id: ev_l3_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '86.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_struq_naive
  evaluation_id: ev_l3_struq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '3.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_struq_ignore
  evaluation_id: ev_l3_struq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '4.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_struq_escape
  evaluation_id: ev_l3_struq_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '4.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_struq_fakecom
  evaluation_id: ev_l3_struq_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '3.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_struq_combined
  evaluation_id: ev_l3_struq_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '16.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_rvr_naive
  evaluation_id: ev_l3_rvr_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_rvr_ignore
  evaluation_id: ev_l3_rvr_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '1.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_rvr_escape
  evaluation_id: ev_l3_rvr_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_rvr_fakecom
  evaluation_id: ev_l3_rvr_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '1.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_rvr_combined
  evaluation_id: ev_l3_rvr_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '0.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_util_squad
  evaluation_id: ev_l3_none_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '83.56'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_util_triviaqa
  evaluation_id: ev_l3_none_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '75.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_none_util_sst2
  evaluation_id: ev_l3_none_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '94.84'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_rvr_util_squad
  evaluation_id: ev_l3_rvr_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '87.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_rvr_util_triviaqa
  evaluation_id: ev_l3_rvr_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '77.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l3_rvr_util_sst2
  evaluation_id: ev_l3_rvr_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '93.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_none_naive
  evaluation_id: ev_q2_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '70.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_ignore
  evaluation_id: ev_q2_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '80.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_escape
  evaluation_id: ev_q2_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '78.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_fakecom
  evaluation_id: ev_q2_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '96.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_combined
  evaluation_id: ev_q2_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '92.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_struq_naive
  evaluation_id: ev_q2_struq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '12.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_struq_ignore
  evaluation_id: ev_q2_struq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '11.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_struq_escape
  evaluation_id: ev_q2_struq_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '11.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_struq_fakecom
  evaluation_id: ev_q2_struq_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '78.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_struq_combined
  evaluation_id: ev_q2_struq_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '82.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_rvr_naive
  evaluation_id: ev_q2_rvr_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '4.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_rvr_ignore
  evaluation_id: ev_q2_rvr_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '2.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_rvr_escape
  evaluation_id: ev_q2_rvr_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '2.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_rvr_fakecom
  evaluation_id: ev_q2_rvr_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '1.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_rvr_combined
  evaluation_id: ev_q2_rvr_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '1.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_util_squad
  evaluation_id: ev_q2_none_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '79.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_none_util_triviaqa
  evaluation_id: ev_q2_none_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '77.22'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_none_util_sst2
  evaluation_id: ev_q2_none_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '94.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_rvr_util_squad
  evaluation_id: ev_q2_rvr_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '88.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_rvr_util_triviaqa
  evaluation_id: ev_q2_rvr_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '78.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_q2_rvr_util_sst2
  evaluation_id: ev_q2_rvr_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '94.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_naive
  evaluation_id: ev_l31_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '64.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_ignore
  evaluation_id: ev_l31_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '77.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_escape
  evaluation_id: ev_l31_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '76.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_fakecom
  evaluation_id: ev_l31_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '85.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_combined
  evaluation_id: ev_l31_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '84.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_struq_naive
  evaluation_id: ev_l31_struq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_struq_ignore
  evaluation_id: ev_l31_struq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '1.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_struq_escape
  evaluation_id: ev_l31_struq_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_struq_fakecom
  evaluation_id: ev_l31_struq_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '46.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_struq_combined
  evaluation_id: ev_l31_struq_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '56.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_rvr_naive
  evaluation_id: ev_l31_rvr_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_rvr_ignore
  evaluation_id: ev_l31_rvr_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_rvr_escape
  evaluation_id: ev_l31_rvr_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_rvr_fakecom
  evaluation_id: ev_l31_rvr_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_rvr_combined
  evaluation_id: ev_l31_rvr_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_util_squad
  evaluation_id: ev_l31_none_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '82.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_util_triviaqa
  evaluation_id: ev_l31_none_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '79.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_none_util_sst2
  evaluation_id: ev_l31_none_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '94.61'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_rvr_util_squad
  evaluation_id: ev_l31_rvr_util_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: SQuAD
  value: '88.22'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_rvr_util_triviaqa
  evaluation_id: ev_l31_rvr_util_triviaqa
  quantity: R2clean
  coverage: direct
  metric_as_printed: TriviaQA
  value: '79.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l31_rvr_util_sst2
  evaluation_id: ev_l31_rvr_util_sst2
  quantity: R2clean
  coverage: direct
  metric_as_printed: SST2
  value: '92.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
comparisons:
- id: c_l3_naive_rvr_none
  left: ev_l3_rvr_naive
  right: ev_l3_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_naive_rvr_struq
  left: ev_l3_rvr_naive
  right: ev_l3_struq_naive
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l3_ignore_rvr_none
  left: ev_l3_rvr_ignore
  right: ev_l3_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ignore_rvr_struq
  left: ev_l3_rvr_ignore
  right: ev_l3_struq_ignore
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l3_escape_rvr_none
  left: ev_l3_rvr_escape
  right: ev_l3_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_escape_rvr_struq
  left: ev_l3_rvr_escape
  right: ev_l3_struq_escape
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l3_fakecom_rvr_none
  left: ev_l3_rvr_fakecom
  right: ev_l3_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fakecom_rvr_struq
  left: ev_l3_rvr_fakecom
  right: ev_l3_struq_fakecom
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l3_combined_rvr_none
  left: ev_l3_rvr_combined
  right: ev_l3_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_combined_rvr_struq
  left: ev_l3_rvr_combined
  right: ev_l3_struq_combined
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l3_util_squad
  left: ev_l3_rvr_util_squad
  right: ev_l3_none_util_squad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_util_triviaqa
  left: ev_l3_rvr_util_triviaqa
  right: ev_l3_none_util_triviaqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l3_util_sst2
  left: ev_l3_rvr_util_sst2
  right: ev_l3_none_util_sst2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q2_naive_rvr_none
  left: ev_q2_rvr_naive
  right: ev_q2_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_naive_rvr_struq
  left: ev_q2_rvr_naive
  right: ev_q2_struq_naive
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_q2_ignore_rvr_none
  left: ev_q2_rvr_ignore
  right: ev_q2_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ignore_rvr_struq
  left: ev_q2_rvr_ignore
  right: ev_q2_struq_ignore
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_q2_escape_rvr_none
  left: ev_q2_rvr_escape
  right: ev_q2_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_escape_rvr_struq
  left: ev_q2_rvr_escape
  right: ev_q2_struq_escape
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_q2_fakecom_rvr_none
  left: ev_q2_rvr_fakecom
  right: ev_q2_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fakecom_rvr_struq
  left: ev_q2_rvr_fakecom
  right: ev_q2_struq_fakecom
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_q2_combined_rvr_none
  left: ev_q2_rvr_combined
  right: ev_q2_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_combined_rvr_struq
  left: ev_q2_rvr_combined
  right: ev_q2_struq_combined
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_q2_util_squad
  left: ev_q2_rvr_util_squad
  right: ev_q2_none_util_squad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q2_util_triviaqa
  left: ev_q2_rvr_util_triviaqa
  right: ev_q2_none_util_triviaqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q2_util_sst2
  left: ev_q2_rvr_util_sst2
  right: ev_q2_none_util_sst2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_naive_rvr_none
  left: ev_l31_rvr_naive
  right: ev_l31_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_naive_rvr_struq
  left: ev_l31_rvr_naive
  right: ev_l31_struq_naive
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l31_ignore_rvr_none
  left: ev_l31_rvr_ignore
  right: ev_l31_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ignore_rvr_struq
  left: ev_l31_rvr_ignore
  right: ev_l31_struq_ignore
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l31_escape_rvr_none
  left: ev_l31_rvr_escape
  right: ev_l31_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_escape_rvr_struq
  left: ev_l31_rvr_escape
  right: ev_l31_struq_escape
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l31_fakecom_rvr_none
  left: ev_l31_rvr_fakecom
  right: ev_l31_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fakecom_rvr_struq
  left: ev_l31_rvr_fakecom
  right: ev_l31_struq_fakecom
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l31_combined_rvr_none
  left: ev_l31_rvr_combined
  right: ev_l31_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_combined_rvr_struq
  left: ev_l31_rvr_combined
  right: ev_l31_struq_combined
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: referencing with filtering (prompting) against StruQ fine-tuning on the same base'
  page: 6
- id: c_l31_util_squad
  left: ev_l31_rvr_util_squad
  right: ev_l31_none_util_squad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_util_triviaqa
  left: ev_l31_rvr_util_triviaqa
  right: ev_l31_none_util_triviaqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_util_sst2
  left: ev_l31_rvr_util_sst2
  right: ev_l31_none_util_sst2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: outperforms prompting baselines, comparable to fine-tuning, ASR near 0% in some scenarios
  location: abstract
  anchor: reducing the ASR to nearly 0% in some scenarios
  jobs:
  - R1
  supported_by:
  - c_l3_naive_rvr_none
  - c_l3_ignore_rvr_none
  - c_l3_escape_rvr_none
  - c_l3_fakecom_rvr_none
  - c_l3_combined_rvr_none
  - c_q2_naive_rvr_none
  - c_q2_ignore_rvr_none
  - c_q2_escape_rvr_none
  - c_q2_fakecom_rvr_none
  - c_q2_combined_rvr_none
  - c_l31_naive_rvr_none
  - c_l31_ignore_rvr_none
  - c_l31_escape_rvr_none
  - c_l31_fakecom_rvr_none
  - c_l31_combined_rvr_none
  - c_l3_naive_rvr_struq
  - c_l3_ignore_rvr_struq
  - c_l3_escape_rvr_struq
  - c_l3_fakecom_rvr_struq
  - c_l3_combined_rvr_struq
  - c_q2_naive_rvr_struq
  - c_q2_ignore_rvr_struq
  - c_q2_escape_rvr_struq
  - c_q2_fakecom_rvr_struq
  - c_q2_combined_rvr_struq
  - c_l31_naive_rvr_struq
  - c_l31_ignore_rvr_struq
  - c_l31_escape_rvr_struq
  - c_l31_fakecom_rvr_struq
  - c_l31_combined_rvr_struq
  page: 1
- id: k2
  text: minimal impact on overall utility
  location: abstract
  anchor: our approach has minimal impact on overall util- ity
  jobs:
  - R2
  supported_by:
  - c_l3_util_squad
  - c_l3_util_triviaqa
  - c_l3_util_sst2
  - c_q2_util_squad
  - c_q2_util_triviaqa
  - c_q2_util_sst2
  - c_l31_util_squad
  - c_l31_util_triviaqa
  - c_l31_util_sst2
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (Inj-SQuAD)
    R2clean: Table 3 (accuracy without attacks)

`moe_arxiv2025` (2504.07467)

In [ ]:
%%writefile reports/records/moe_arxiv2025.yaml
report:
  id: moe_arxiv2025
  slug: '2504.07467'
  title: Defense against Prompt Injection Attacks via Mixture of Encodings
  first_author: Ruiyi Zhang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2504.07467'
  version_read: arXiv v1
  first_date: '2025-04-10'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'Sections 3 to 5: the No Defense, Base64 and Ours rows of Table 1 (all four datasets) and of Table 2 for MMLU, Squad, MGSM and SamSum; the abstract claims (Datamark, Ignoring and Caesar rows, the other
    Table 2 columns and Appendices G and H not coded)'
defences:
- id: moe_gpt4
  name: Mixture of encodings
  origin: this_report
  base_checkpoint: GPT-4 (turbo-2024-04-09)
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - 'three calls: external text in plain form, in Base64 and in a Caesar cipher (shift 3)'
  - 'classification: label with the highest summed probability over the three answers'
  - 'generation: a fourth call with a meta-prompt that writes the answer from the three responses'
  layout: user prompt followed by the external text (plain, Base64 or Caesar), one call each
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We first input both prompts P1 and P2 from
  pages:
    components: 3
    layout: 3
  signal_dependence_detail: removing the encodings leaves the No Defense prompt; Base64 alone and Caesar alone are run as baselines
- id: b64_gpt4
  name: Base64 defence (spotlighting by encoding)
  origin: other_report
  base_checkpoint: GPT-4 (turbo-2024-04-09)
  axis1:
  - encodings
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: 'P2: [User Prompt] + Base64(External Text)'
- id: moe_gpt4o
  name: Mixture of encodings
  origin: this_report
  base_checkpoint: GPT-4o (2024-05-13)
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - 'three calls: external text in plain form, in Base64 and in a Caesar cipher (shift 3)'
  - 'classification: label with the highest summed probability over the three answers'
  - 'generation: a fourth call with a meta-prompt that writes the answer from the three responses'
  layout: user prompt followed by the external text (plain, Base64 or Caesar), one call each
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We first input both prompts P1 and P2 from
  pages:
    components: 3
    layout: 3
  signal_dependence_detail: removing the encodings leaves the No Defense prompt; Base64 alone and Caesar alone are run as baselines
- id: b64_gpt4o
  name: Base64 defence (spotlighting by encoding)
  origin: other_report
  base_checkpoint: GPT-4o (2024-05-13)
  axis1:
  - encodings
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: 'P2: [User Prompt] + Base64(External Text)'
evaluations:
- id: ev_gpt4_none_email
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_table
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_abstract
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_code
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_mmlu
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_squad
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_mgsm
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4_none_samsum
  defence_id: none:GPT-4 (turbo-2024-04-09)
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4_b64_email
  defence_id: b64_gpt4
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_b64_table
  defence_id: b64_gpt4
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_b64_abstract
  defence_id: b64_gpt4
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_b64_code
  defence_id: b64_gpt4
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_b64_mmlu
  defence_id: b64_gpt4
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
- id: ev_gpt4_b64_squad
  defence_id: b64_gpt4
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
- id: ev_gpt4_b64_mgsm
  defence_id: b64_gpt4
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
- id: ev_gpt4_b64_samsum
  defence_id: b64_gpt4
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
- id: ev_gpt4_moe_email
  defence_id: moe_gpt4
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_moe_table
  defence_id: moe_gpt4
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_moe_abstract
  defence_id: moe_gpt4
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_moe_code
  defence_id: moe_gpt4
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4_moe_mmlu
  defence_id: moe_gpt4
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
- id: ev_gpt4_moe_squad
  defence_id: moe_gpt4
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
- id: ev_gpt4_moe_mgsm
  defence_id: moe_gpt4
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
- id: ev_gpt4_moe_samsum
  defence_id: moe_gpt4
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
- id: ev_gpt4o_none_email
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_table
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_abstract
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_code
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_mmlu
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_squad
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_mgsm
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4o_none_samsum
  defence_id: none:GPT-4o (2024-05-13)
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpt4o_b64_email
  defence_id: b64_gpt4o
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_b64_table
  defence_id: b64_gpt4o
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_b64_abstract
  defence_id: b64_gpt4o
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_b64_code
  defence_id: b64_gpt4o
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_b64_mmlu
  defence_id: b64_gpt4o
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
- id: ev_gpt4o_b64_squad
  defence_id: b64_gpt4o
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
- id: ev_gpt4o_b64_mgsm
  defence_id: b64_gpt4o
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
- id: ev_gpt4o_b64_samsum
  defence_id: b64_gpt4o
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
- id: ev_gpt4o_moe_email
  defence_id: moe_gpt4o
  benchmark: BIPIA subset, Email (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 11,250
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_moe_table
  defence_id: moe_gpt4o
  benchmark: BIPIA subset, Table (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_moe_abstract
  defence_id: moe_gpt4o
  benchmark: BIPIA subset, Abstract (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 22,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_moe_code
  defence_id: moe_gpt4o
  benchmark: BIPIA subset, Code (50 attack types)
  attacker_type: static
  attacker_access: black_box
  judge: not stated (BIPIA attacks)
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 3
  sample_size: 7,500
  decoding: not reported
  pages:
    benchmark: 4
- id: ev_gpt4o_moe_mmlu
  defence_id: moe_gpt4o
  benchmark: MMLU (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14 K
  decoding: not reported
- id: ev_gpt4o_moe_squad
  defence_id: moe_gpt4o
  benchmark: Squad (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 10.6 K
  decoding: not reported
- id: ev_gpt4o_moe_mgsm
  defence_id: moe_gpt4o
  benchmark: MGSM (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 1.3 K
  decoding: not reported
- id: ev_gpt4o_moe_samsum
  defence_id: moe_gpt4o
  benchmark: SamSum (helpfulness benchmark, no attack)
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric not stated
  layout_same_as_defence: 'yes'
  measured_by: moe_arxiv2025
  page: 4
  sample_size: 14.7 K
  decoding: not reported
results:
- id: r_gpt4_none_email
  evaluation_id: ev_gpt4_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '14.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_none_table
  evaluation_id: ev_gpt4_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '34.52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_none_abstract
  evaluation_id: ev_gpt4_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '25.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_none_code
  evaluation_id: ev_gpt4_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '1.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_none_mmlu
  evaluation_id: ev_gpt4_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '83.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_none_squad
  evaluation_id: ev_gpt4_none_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '43.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_none_mgsm
  evaluation_id: ev_gpt4_none_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '38.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_none_samsum
  evaluation_id: ev_gpt4_none_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '41.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_b64_email
  evaluation_id: ev_gpt4_b64_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '3.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_b64_table
  evaluation_id: ev_gpt4_b64_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '10.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_b64_abstract
  evaluation_id: ev_gpt4_b64_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '8.66'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_b64_code
  evaluation_id: ev_gpt4_b64_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '0.15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_b64_mmlu
  evaluation_id: ev_gpt4_b64_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '44.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_b64_squad
  evaluation_id: ev_gpt4_b64_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '43.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_b64_mgsm
  evaluation_id: ev_gpt4_b64_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '19.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_b64_samsum
  evaluation_id: ev_gpt4_b64_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '37.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_moe_email
  evaluation_id: ev_gpt4_moe_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '1.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_moe_table
  evaluation_id: ev_gpt4_moe_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '3.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_moe_abstract
  evaluation_id: ev_gpt4_moe_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '6.79'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_moe_code
  evaluation_id: ev_gpt4_moe_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '0.07'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4_moe_mmlu
  evaluation_id: ev_gpt4_moe_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '77.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_moe_squad
  evaluation_id: ev_gpt4_moe_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '43.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_moe_mgsm
  evaluation_id: ev_gpt4_moe_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '36.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4_moe_samsum
  evaluation_id: ev_gpt4_moe_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '38.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_none_email
  evaluation_id: ev_gpt4o_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '12.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_none_table
  evaluation_id: ev_gpt4o_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '36.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_none_abstract
  evaluation_id: ev_gpt4o_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '26.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_none_code
  evaluation_id: ev_gpt4o_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '7.59'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_none_mmlu
  evaluation_id: ev_gpt4o_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '79.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_none_squad
  evaluation_id: ev_gpt4o_none_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '43.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_none_mgsm
  evaluation_id: ev_gpt4o_none_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '53.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_none_samsum
  evaluation_id: ev_gpt4o_none_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '41.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_b64_email
  evaluation_id: ev_gpt4o_b64_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '1.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_b64_table
  evaluation_id: ev_gpt4o_b64_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '1.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_b64_abstract
  evaluation_id: ev_gpt4o_b64_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '5.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_b64_code
  evaluation_id: ev_gpt4o_b64_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_b64_mmlu
  evaluation_id: ev_gpt4o_b64_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_b64_squad
  evaluation_id: ev_gpt4o_b64_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '37.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_b64_mgsm
  evaluation_id: ev_gpt4o_b64_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '5.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_b64_samsum
  evaluation_id: ev_gpt4o_b64_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '35.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_moe_email
  evaluation_id: ev_gpt4o_moe_email
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Email
  value: '1.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_moe_table
  evaluation_id: ev_gpt4o_moe_table
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Table
  value: '1.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_moe_abstract
  evaluation_id: ev_gpt4o_moe_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Abstract
  value: '1.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_moe_code
  evaluation_id: ev_gpt4o_moe_code
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Code
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_gpt4o_moe_mmlu
  evaluation_id: ev_gpt4o_moe_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '75.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_moe_squad
  evaluation_id: ev_gpt4o_moe_squad
  quantity: R2clean
  coverage: direct
  metric_as_printed: Squad
  value: '42.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_moe_mgsm
  evaluation_id: ev_gpt4o_moe_mgsm
  quantity: general_utility
  coverage: direct
  metric_as_printed: MGSM
  value: '52.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt4o_moe_samsum
  evaluation_id: ev_gpt4o_moe_samsum
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SamSum
  value: '39.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
comparisons:
- id: c_gpt4_email_moe_none
  left: ev_gpt4_moe_email
  right: ev_gpt4_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4_email_moe_b64
  left: ev_gpt4_moe_email
  right: ev_gpt4_b64_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4_table_moe_none
  left: ev_gpt4_moe_table
  right: ev_gpt4_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4_table_moe_b64
  left: ev_gpt4_moe_table
  right: ev_gpt4_b64_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4_abstract_moe_none
  left: ev_gpt4_moe_abstract
  right: ev_gpt4_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4_abstract_moe_b64
  left: ev_gpt4_moe_abstract
  right: ev_gpt4_b64_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4_code_moe_none
  left: ev_gpt4_moe_code
  right: ev_gpt4_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4_code_moe_b64
  left: ev_gpt4_moe_code
  right: ev_gpt4_b64_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4_mmlu_moe_none
  left: ev_gpt4_moe_mmlu
  right: ev_gpt4_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4_mmlu_moe_b64
  left: ev_gpt4_moe_mmlu
  right: ev_gpt4_b64_mmlu
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4_squad_moe_none
  left: ev_gpt4_moe_squad
  right: ev_gpt4_none_squad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4_squad_moe_b64
  left: ev_gpt4_moe_squad
  right: ev_gpt4_b64_squad
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4_mgsm_moe_none
  left: ev_gpt4_moe_mgsm
  right: ev_gpt4_none_mgsm
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4_mgsm_moe_b64
  left: ev_gpt4_moe_mgsm
  right: ev_gpt4_b64_mgsm
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4_samsum_moe_none
  left: ev_gpt4_moe_samsum
  right: ev_gpt4_none_samsum
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4_samsum_moe_b64
  left: ev_gpt4_moe_samsum
  right: ev_gpt4_b64_samsum
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4o_email_moe_none
  left: ev_gpt4o_moe_email
  right: ev_gpt4o_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4o_email_moe_b64
  left: ev_gpt4o_moe_email
  right: ev_gpt4o_b64_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4o_table_moe_none
  left: ev_gpt4o_moe_table
  right: ev_gpt4o_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4o_table_moe_b64
  left: ev_gpt4o_moe_table
  right: ev_gpt4o_b64_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4o_abstract_moe_none
  left: ev_gpt4o_moe_abstract
  right: ev_gpt4o_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4o_abstract_moe_b64
  left: ev_gpt4o_moe_abstract
  right: ev_gpt4o_b64_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4o_code_moe_none
  left: ev_gpt4o_moe_code
  right: ev_gpt4o_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_gpt4o_code_moe_b64
  left: ev_gpt4o_moe_code
  right: ev_gpt4o_b64_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 3
- id: c_gpt4o_mmlu_moe_none
  left: ev_gpt4o_moe_mmlu
  right: ev_gpt4o_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4o_mmlu_moe_b64
  left: ev_gpt4o_moe_mmlu
  right: ev_gpt4o_b64_mmlu
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4o_squad_moe_none
  left: ev_gpt4o_moe_squad
  right: ev_gpt4o_none_squad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4o_squad_moe_b64
  left: ev_gpt4o_moe_squad
  right: ev_gpt4o_b64_squad
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4o_mgsm_moe_none
  left: ev_gpt4o_moe_mgsm
  right: ev_gpt4o_none_mgsm
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4o_mgsm_moe_b64
  left: ev_gpt4o_moe_mgsm
  right: ev_gpt4o_b64_mgsm
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
- id: c_gpt4o_samsum_moe_none
  left: ev_gpt4o_moe_samsum
  right: ev_gpt4o_none_samsum
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_gpt4o_samsum_moe_b64
  left: ev_gpt4o_moe_samsum
  right: ev_gpt4o_b64_samsum
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the mixture (plain, Base64 and Caesar answers combined) against Base64 alone
  page: 4
claims:
- id: k1
  text: one of the lowest attack success rates
  location: abstract
  anchor: achieves one of the low- est attack success rates under prompt injec- tion attacks
  jobs:
  - R1
  supported_by:
  - c_gpt4_email_moe_none
  - c_gpt4_table_moe_none
  - c_gpt4_abstract_moe_none
  - c_gpt4_code_moe_none
  - c_gpt4o_email_moe_none
  - c_gpt4o_table_moe_none
  - c_gpt4o_abstract_moe_none
  - c_gpt4o_code_moe_none
  page: 1
- id: k2
  text: high performance across all NLP tasks, better than other encoding defences
  location: abstract
  anchor: while maintaining high performance across all NLP tasks
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_gpt4_mmlu_moe_b64
  - c_gpt4_squad_moe_b64
  - c_gpt4_mgsm_moe_b64
  - c_gpt4_samsum_moe_b64
  - c_gpt4o_mmlu_moe_b64
  - c_gpt4o_squad_moe_b64
  - c_gpt4o_mgsm_moe_b64
  - c_gpt4o_samsum_moe_b64
  page: 1
  contradicted_by:
  - r_gpt4_moe_mmlu
  - r_gpt4o_moe_mmlu
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: Table 2 (Squad; SamSum as a summary proxy)
    general_utility: Table 2 (MMLU, MGSM)

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04a"
IDS = ['struq_arxiv2024', 'ih_arxiv2024', 'ise_iclr2025', 'metasecalign_arxiv2025', 'defensivetokens_aisec2025', 'jatmo_arxiv2023', 'bipia_kdd2025', 'cacheprune_arxiv2025', 'drip_arxiv2025', 'fath_arxiv2024', 'rvr_arxiv2025', 'moe_arxiv2025']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; three extra checks

In [ ]:
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual = [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept = 0
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val}; chart values {fig_kept}; anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in man.values() if m.get("decision") == "include")
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "chart_values_checked", "anchors_checked",
                                              "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values, {tot['chart_values_checked']} chart values and "
         f"{tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: 0",
         "next: batch 04b"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04a extraction batch a: twelve included studies coded from their PDFs, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)